In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import timedelta
import sys
sys.path.append("../aws_utils")
import aws_utils

#env = aws_utils.get_env_prefix() # we will only keep this in the final prod notebook & replace all env2

env = aws_utils.get_env_prefix()
#env = 'p' #--change here!!! when we go to prod it will be one env. We cannot run for the main tables in dev because it is not in synch. Only used for GETTING data from there we do not write on tables.
env2 = aws_utils.get_env_prefix() # same as env - previously was separate for export tables

In [0]:
# Create a widget (can be 'normal' or 'mom')--manual=normal run // mom =middle of the month run.(for missing account data that just migrated)
# #ADD IT ON FLOW TO GET FORM THERE AUTOMATICALLY!!
#dbutils.widgets.text("run_type", "mom")
# Create the widget with a default value  
dbutils.widgets.text("run_type", "normal")

# Get the widget value
run_type = dbutils.widgets.get("run_type")

#`ref_calendar` is loaded into a DataFrame
ref_calendar = spark.table(f"`{env}-cdh`.reference.ref_calendar")

# Filter for the current date--> will be put to run for the running date
today_date = spark.sql("SELECT current_date() as today").collect()[0]['today']
current_date = (today_date - timedelta(days=1)).strftime('%Y-%m-%d')
#current_date='2025-10-28'#-- for manual run

# ---CHECK : If date is not a business day, get the previous one ---
calendar_check = ref_calendar.filter(ref_calendar.CalendarDate == current_date).collect()
if not calendar_check:
    raise ValueError(f"Error: The date {current_date} was not found in reference.ref_calendar.")

if calendar_check[0]["IsBusinessDay"] == 0:
    current_date = calendar_check[0]["PreviousBusinessDay"]
    print(f"Initial date was not a business day. Using previous business day: {current_date}")
# ---End of Check---

filtered_calendar = ref_calendar.filter(ref_calendar.CalendarDate == current_date)
filtered_calendar.createOrReplaceTempView("filtered_calendar")

#NOTE_ : we need the    CALENDAR date to be a Business Date so that we get results(no weekends or days off)!!
if run_type == "normal":
# TIRESIAS.PARAMETER
      param_dates_query =  """
                              SELECT 
                              CalendarDate 
                              ,CAST(dateadd(MONTH,-1,c.FirstDateOfMonth) AS DATE)  AS som_Date  -- start of previous month
                              ,CAST(c.PreviousMonthLastDate AS DATE)  AS eom_Date-- end of previous month
                              ,CAST(c.PreviousMonthLastWorkingDate AS DATE)   AS eom_lst_wrk_date --end of  previosu month WORKING date  so that theres is flow
                              ,CAST(dateadd(DAY,-1,dateadd(MONTH,-1,c.FirstDateOfMonth)) AS DATE)  AS previous_eom_Date--last date of previous export month(another month behind)
                              FROM filtered_calendar C
                        """
      

elif run_type == "mom":

      param_dates_query = """
                              SELECT 
                              CalendarDate
                              ,CAST(c.FirstDateOfMonth AS DATE)  AS som_Date
                              ,CalendarDate  AS eom_Date
                              ,CalendarDate  AS eom_lst_wrk_date
                              ,CAST(dateadd(DAY,-1,dateadd(MONTH,-1,c.FirstDateOfMonth)) AS DATE)  AS previous_eom_Date
                              FROM filtered_calendar C
                        """
      

else:
    raise ValueError("Invalid run_type. Must be 'normal' or 'mom'.")

param_dates = spark.sql(param_dates_query)
param_dates.createOrReplaceTempView("param_dates")

# TIRESIAS.portfolio
portfolio = spark.sql(
    f"""
                         SELECT
                         p.PortfolioId
                        ,p.PortfolioName
                        ,cast(p.AssignmentDate AS DATE)
                        ,p.TiresiasName
                        ,p.TiresiasCode
                        ,p.prefix
                        ,pp.eom_Date AS eomDate
                        ,pp.eom_lst_wrk_date AS eomWrkDate
                        ,pp.som_Date AS somDate
                        ,cast(pp.CalendarDate AS DATE) AS CalendarDate
                        ,CASE WHEN p.originBank='ALPHA BANK'  THEN  '014'
                              WHEN p.originBank='ATTICA BANK' THEN '016'
                              WHEN p.originBank='NBG' THEN '011'
                              WHEN p.originBank='PIRAEUS BANK' THEN '017'
                              WHEN p.originBank='NBG, ALPHA BANK' THEN '014'
                              WHEN p.originBank='ALPHA BANK, EUROBANK, NBG' THEN '014, 026, 011'
                              WHEN p.originBank='UCI SPAIN' THEN '058'
                             ---to be completed with more if more bank examples come
                        END AS OriginBankCode
                        ,CASE WHEN p.AssetOwner ='Hoist Finance AB (PUBL)' THEN 2
                              ELSE 1 END AS ExportPartition
                        ,processSettlementFlag
                        FROM `{env}-cdh`.primary.portfolio p
                        CROSS JOIN param_dates  pp
                        WHERE COALESCE(unit_code_profits,0)>0 
                        AND tiresiascode IS NOT null
                        AND businessdate=pp.eom_lst_wrk_date               
                        """
)

portfolio.createOrReplaceTempView("portfolio")
# --- CHECK: Check if portfolio DataFrame is empty then all else will be empty so crash ---
if spark.sql(f"SELECT COUNT(*) FROM portfolio").collect()[0][0] == 0:
    raise ValueError("Error: The 'portfolio' DataFrame is empty. No portfolio data found for the selected date.")
# ---End of Check ---

display(param_dates)
#display(portfolio)

In [0]:
#temps needed to fill loandata
#--------------------------------------------------------------
# Open Accounts  
d_exposure_perimeter = spark.sql( f"""
                        
                            SELECT e.*
                            FROM `{env}-cdh`.derivative.dexposure e
                            INNER JOIN portfolio p
                            ON    e.portfolioname=p.PortfolioName 
                            WHERE e.BusinessDate =p.eomWrkDate
                            AND e.isactiveflag = TRUE
                            AND e.closedDate  IS NULL --NON CLOSED
                            AND e.openDate !=COALESCE(e.closedDate,'9999-12-31')-- wrong cases that opened and closed same day
                                         
                        """
)
d_exposure_perimeter.createOrReplaceTempView("d_exposure_perimeter")
#--------------------------------------------------------------
# Accounts that closed this month before the end of month that the code runs for
d_closed_perimeter = spark.sql( f"""
                        SELECT * 
                        FROM (
                            SELECT e.*
                                    ,ROW_NUMBER() OVER (PARTITION BY e.Exposurekey ORDER BY e.BusinessDate ASC  ) AS rn
                            FROM `{env}-cdh`.derivative.dexposure e
                            INNER JOIN portfolio p
                            ON    e.portfolioname=p.PortfolioName 
                            WHERE e.BusinessDate BETWEEN p.somDate AND p.eomDate 
                            AND e.isactiveflag = FALSE
                            AND e.closedDate BETWEEN p.somDate AND p.eomWrkDate--closed in this month
                            AND e.openDate !=COALESCE(e.closedDate,'9999-12-31')-- wrong cases that opened and closed same day
                        ) C WHERE rn=1                    
                        """
)

d_closed_perimeter.createOrReplaceTempView("d_closed_perimeter")

closed_perimeter = spark.sql( f"""
                        SELECT * 
                        FROM (
                                SELECT e.*
                                    , ROW_NUMBER() OVER (PARTITION BY  e.Exposurekey ,e.partyrelationshiptype,cdi  ORDER BY e.BusinessDate ASC  ) AS rn
                                FROM `{env}-cdh`.primary.exposure e
                                INNER JOIN portfolio p
                                ON    e.portfolioname=p.PortfolioName
                                WHERE e.BusinessDate BETWEEN p.somDate AND p.eomDate
                                AND e.isactiveflag = FALSE
                                AND SourceSystem = 'profits'
                        ) C WHERE rn=1                    
                        """
)

closed_perimeter.createOrReplaceTempView("closed_perimeter")

#--------------------------------------------------------------
# dpmaymentschedule
installments = spark.sql( f"""
                        
                            SELECT e.Exposurekey, COUNT( distinct installmentnumber) as NumberofInstallments
                            FROM `{env}-cdh`.derivative.dpaymentschedule p
                            INNER JOIN d_exposure_perimeter e
                            ON    e.Exposurekey=p.Exposurekey 
                            AND   e.BusinessDate =p.BusinessDate
                            GROUP BY e.Exposurekey

                            UNION

                            SELECT e.Exposurekey, COUNT(distinct installmentnumber)
                            FROM `{env}-cdh`.derivative.dpaymentschedule p
                            INNER JOIN closed_perimeter e
                            ON    e.Exposurekey=p.Exposurekey 
                            AND   e.BusinessDate =p.BusinessDate
                            WHERE NOT EXISTS (SELECT 1 FROM d_exposure_perimeter dep WHERE dep.Exposurekey = e.Exposurekey) -- Exclude cases that closed and reopened in the same month
                            GROUP BY e.Exposurekey
                                         
                        """
)
installments.createOrReplaceTempView("installments")
#--------------------------------------------------------------
# payment
payment = spark.sql( f"""
                    SELECT DISTINCT
                    -- pay.exposurekey 
                    MAX(pay.ActualDate)  AS last_payment_date
                    ,SUM(pay.Amount) AS total_payment_amount
                    ,SUM(CASE WHEN pay.ActualDate BETWEEN p.somDate AND p.eomDate THEN pay.Amount END)  
                     AS prv_month_payment_amount
                    ,pay.SourceExposureKey
                    ,p.Prefix
                  
                    FROM `{env}-cdh`.derivative.dcollection pay
                    INNER JOIN portfolio p
                    ON    pay.portfolioname=p.PortfolioName 
                    WHERE 1=1
                    AND   pay.businessdate=(SELECT MAX(prt.businessdate) FROM `{env}-cdh`.primary.portfolio prt WHERE prt.PortfolioName=p.PortfolioName)--MAX portfolio businessdate
                    AND   pay.ActualDate BETWEEN p.AssignmentDate AND p.eomDate
                    AND   pay.finalReason='Matched'  
                    GROUP BY pay.SourceExposureKey , p.Prefix    
                

                        """
)
 
payment.createOrReplaceTempView("payment")

payment2 = spark.sql( f"""
                    SELECT    normalized_loanbankcode,
                        MAX(last_payment_date) AS last_payment_date,
                        SUM(total_payment_amount) AS total_payment_amount,
                        SUM(prv_month_payment_amount) AS prv_month_payment_amount
                    FROM (
                        SELECT
                            *,
                            CASE
                                WHEN LEFT(SourceExposureKey, 3) = Prefix THEN SourceExposureKey
                                ELSE CONCAT(Prefix, SourceExposureKey)
                            END AS normalized_loanbankcode
                        FROM payment
                    ) sub
                    GROUP BY normalized_loanbankcode
                

                        """
)
 
payment2.createOrReplaceTempView("payment2")


#--------------------------------------------------------------
# underlaw
underlaw = spark.sql( f"""
                        SELECT DISTINCT cdi,underLawType
                        FROM `{env}-cdh`.derivative.dunderlaw law --`{env}-cdh`.primary.underlaw law
                        INNER JOIN portfolio p
                        ON    p.portfolioName=LAW.portfolioName
                        WHERE law.businessdate= p.eomWrkDate
                        AND   law.inlawflag=1
                        AND  law.underLawType='N3869'                    
                        """
)
underlaw.createOrReplaceTempView("underlaw")

#--------------------------------------------------------------
#master-slave 
#trying to make tiresias.slaveaccounts from on prem

l1_l2_sub = spark.sql( f""" 
                        SELECT DISTINCT trim(account_number) as account_number,trim(DEBIT_OPEN_ACCOUNT) as DEBIT_OPEN_ACCOUNT 
                        FROM  `{env}-cdh`.landed.profits_lns_ext_subsidy_hd l
                        CROSS JOIN param_dates param
                        WHERE l.businessdate=param.eom_lst_wrk_date 
                            """
)
l1_l2_sub.createOrReplaceTempView("l1_l2_sub")

RestructuringWriteoff = spark.sql(f""" SELECT DISTINCT
                                              PortfolioName
                                              ,WriteOffExposureKey--SLAVE
                                             ,RestructuringExposureKey--MASTER
                                        FROM `{env}-cdh`.primary.RestructuringWriteoff restr--FROM  `{env}-cdh`.derivative.drestructuringwriteoff restr
                                        CROSS JOIN param_dates param
                                        WHERE restr.businessdate=param.eom_lst_wrk_date 
"""
) 

RestructuringWriteoff.createOrReplaceTempView("RestructuringWriteoff")

master_slave_Accounts_only = spark.sql( f"""
                            SELECT  DISTINCT e.portfolioName
                                    ,RelatedEntityKey as slaveAccount
                                    ,entityKey as MasterAccount
                                    ,e.EntityRelationshipType
                                    ,e.EntityRelationshipSubType 
     
                            FROM `{env}-cdh`.derivativeactive.dentityRelationships e 
                            INNER JOIN portfolio p
                            ON p.portfolioname=e.portfolioname
                            WHERE e.EntityRelationshipType ='Master - Slave Rltnp'
                           -- AND e.EntityRelationshipSubType = "WDN"
                            AND e.RelatedEntityTYPE ='Exposure' 
                            AND e.EntityType='Exposure'
                            AND e.businessDate=p.eomWrkDate 
                        UNION 
                            --CLOSED
                            SELECT  DISTINCT e.portfolioName
                                ,RelatedEntityKey as slaveAccount
                                ,entityKey as MasterAccount
                                ,e.EntityRelationshipType
                                ,e.EntityRelationshipSubType
                               
                            FROM `{env}-cdh`.derivativeactive.dentityRelationships e  --`{env}-cdh`.primary.entityRelationships e
                            INNER JOIN portfolio p
                            ON p.portfolioname=e.portfolioname
                            INNER JOIN d_closed_perimeter c--closed_perimeter 
                            ON c.exposurekey=e.RelatedEntityKey
                            AND e.businessDate=c.businessDate
                            WHERE e.EntityRelationshipType ='Master - Slave Rltnp'
                          --  AND e.EntityRelationshipSubType = "WDN"
                            AND e.RelatedEntityTYPE ='Exposure' 
                            AND e.EntityType='Exposure'

                            """
)
master_slave_Accounts_only.createOrReplaceTempView("master_slave_Accounts_only")


master_slave_Accounts_UNION = spark.sql( f"""
                        SELECT  '1'  as relationshipType_rnk--ranking is based on assetrelationshiptypeid ASC ranking form on prem query
                                ,slaveAccount
                                ,MasterAccount
                                ,EntityRelationshipType
                                ,EntityRelationshipSubType
                                --,source
                        FROM    master_slave_Accounts_only
                        WHERE SUBSTRING(slaveAccount, 1, 3) RLIKE '^[A-Za-z]+$'
                        
                        UNION

                        SELECT  '3'  as relationshipType_rnk
                                ,DEBIT_OPEN_ACCOUNT
                                ,account_number
                                ,'L1-L2 Subsidy Account'
                                ,CAST(NULL AS STRING)
                            --    ,'DWH'
                        FROM    l1_l2_sub
                        WHERE SUBSTRING(DEBIT_OPEN_ACCOUNT, 1, 3) RLIKE '^[A-Za-z]+$'

                        UNION

                        SELECT  '2'  as relationshipType_rnk
                                ,WriteOffExposureKey--SLAVE
                                ,RestructuringExposureKey--MASTER
                                ,'Restructured Loan - Writeoff Rltnp'
                                ,CAST(NULL AS STRING)
                               -- ,'DWH'
                        FROM    RestructuringWriteoff
                        WHERE SUBSTRING(WriteOffExposureKey, 1, 3) RLIKE '^[A-Za-z]+$'
                            """
)

master_slave_Accounts_UNION.createOrReplaceTempView("master_slave_Accounts_UNION")


master_slave_Accounts = spark.sql( f"""
                        SELECT  ms.*
                        FROM master_slave_Accounts_UNION ms
                        INNER JOIN `{env}-cdh`.derivative.dexposure e
                        ON    e.exposurekey=ms.slaveAccount
                        INNER JOIN portfolio p
                        ON     p.portfolioname=e.portfolioname 
                        WHERE  e.businessdate=p.eomWrkDate
                        UNION 
                        SELECT  ms.*
                        FROM master_slave_Accounts_UNION ms
                        INNER JOIN d_closed_perimeter e
                        ON    e.exposurekey=ms.slaveAccount
                        INNER JOIN portfolio p
                        ON     p.portfolioname=e.portfolioname 
                        WHERE  e.businessdate=p.eomWrkDate

                            """
)

master_slave_Accounts.createOrReplaceTempView("master_slave_Accounts")

master_slave_Accounts_ranked = spark.sql( f"""
                        select *,substring(MasterAccount,4,100),ROW_NUMBER() OVER(PARTITION BY SlaveAccount ORDER BY relationshipType_rnk ASC,substring(MasterAccount,4,100) ASC) AS rn
                        FROM    master_slave_Accounts
                            """
)

master_slave_Accounts_ranked.createOrReplaceTempView("master_slave_Accounts_ranked")
#--------------------------------------------------------------
#-------------------modif_pee fields
restructuring = spark.sql(f"""
                          SELECT * 
                          FROM `{env}-cdh`.primary.restructuring r
                          INNER JOIN portfolio p
                          ON     p.portfolioname=r.portfolioname 
                          WHERE  r.businessdate BETWEEN p.somDate AND p.eomWrkDate 
                          AND SourceSystem = 'profits'
                          """)

restructuring.createOrReplaceTempView("restructuring")

restructuring_2 = spark.sql(f"""
                           SELECT DISTINCT OldExposureKey AS OldAccountNumber, MAX(RestructuringDate) AS ImplementationDate
                           FROM restructuring 
                           GROUP BY OldExposureKey
                          """)

restructuring_2.createOrReplaceTempView("restructuring_2")

settlement_ranked = spark.sql(f"""
                          SELECT DISTINCT S.BUSINESSDATE
                                        ,s.exposurekey 
                                        ,s.CDI
                                        ,s.SettlementKey 
                                        ,s.SettlementStatusCode 
                                        ,CAST(s.ExpiredDate AS DATE) AS ExpiredDate
                                        ,s.OverdueDate 
                                        ,CAST(s.implementationDate AS DATE) AS implementationDate
                                        ,ROW_NUMBER() OVER (PARTITION BY exposurekey, businessdate ORDER BY CreatedDate) as rn--needs cdi?
                           FROM `{env}-cdh`.primary.SettlementExposure s
                           INNER JOIN portfolio p
                           ON     p.portfolioname=s.portfolioname 
                           WHERE  s.BusinessDate= p.eomWrkDate 
                          """)

settlement_ranked.createOrReplaceTempView("settlement_ranked")



#--------------------------------------------------------------
#-------------------new ΑΓΟΡΑΣΤΗΣ ΠΡΩΤΗ ΦΟΡΑ fields
GovernmentGuaranteeFlag = spark.sql( f"""
                                SELECT DISTINCT  
                                                SUBSTRING(e.ExposureKey,4,99)
                                                ,e.ExposureKey
                                FROM `{env}-cdh`.derivative.dexposure e
                                INNER JOIN portfolio p
                                ON     p.portfolioname=e.portfolioname 
                                WHERE  e.businessdate=p.eomWrkDate
                                AND    e.IsActiveFlag=TRUE
                                AND    E.IsGuaranteeFlag=TRUE                  
                        """
)
GovernmentGuaranteeFlag.createOrReplaceTempView("GovernmentGuaranteeFlag")

syndicatedFlagLoan = spark.sql( f"""
                            SELECT DISTINCT 
                                    e.exposurekey AS account_number
                                    ,CASE WHEN e.AgreementKey LIKE '%-%' THEN SUBSTRING(e.AgreementKey,15,100)
                                             ELSE SUBSTRING(e.AgreementKey,4,99)
                                    END AS `ΦΥΣΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                                    ,e.AgreementKey AS `ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                                    ,sf.AgreementNumber AS `ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`
                            FROM `{env}-cdh`.primary.syndicatedfacility sf
                            INNER JOIN  portfolio p
                            ON p.portfolioname=sf.portfolioname AND sf.businessdate=p.eomWrkDate
                            INNER JOIN  `{env}-cdh`.derivative.dexposure e 
                            ON  sf.exposurekey=e.exposurekey
                            AND e.businessdate=sf.businessdate                 
                        """
)
syndicatedFlagLoan.createOrReplaceTempView("syndicatedFlagLoan")

syndicatedFlagAmount = spark.sql( f"""
                                SELECT DISTINCT 
                                    sfl.`ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                                    ,e.ExposureKey
                                    ,e.InitialAssignmentAmount
                                FROM   `{env}-cdh`.derivative.dexposure e 
                                INNER JOIN portfolio p
                                ON p.portfolioname=e.portfolioname AND e.businessdate=p.eomWrkDate
                                INNER JOIN syndicatedFlagLoan sfl ON sfl.`ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`=e.AgreementKey                
                        """
)
syndicatedFlagAmount.createOrReplaceTempView("syndicatedFlagAmount")

syndicatedFlagAmountSum = spark.sql( f"""
                                    SELECT  `ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                                            ,SUM(InitialAssignmentAmount) AS `ΣΥΝΟΛΙΚΟ ΕΓΚΡΙΘΕΝ ΠΟΣΟ ΤΗΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`
                                    FROM    syndicatedFlagAmount
                                    GROUP BY `ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`               
                        """
)
syndicatedFlagAmountSum.createOrReplaceTempView("syndicatedFlagAmountSum")

koinopraktika = spark.sql( f"""
                        SELECT DISTINCT  
                                k.AgreementKey AS `ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                               ,k.ExposureKey
                               ,`% συμμετοχής του SPV είναι`
	                          , '1' AS `Δείκτης ποσοστού συμμετοχής κοινοπρακτικού δανείου`
                        FROM   `{env2}-cdh-delivery`.tiresias.koinopraktika k
                        INNER JOIN syndicatedFlagLoan sfl
                        ON  sfl.`ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`=k.AgreementKey
                        AND sfl.account_number=k.ExposureKey
                        WHERE k.`% συμμετοχής του SPV είναι`='Πλειοψηφικό'
                        """
)
koinopraktika.createOrReplaceTempView("koinopraktika")

syndicatedFlagFields = spark.sql( f"""
                        SELECT DISTINCT  
                                 sfl.account_number
                                ,sfa.`ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                                ,`ΣΥΝΟΛΙΚΟ ΕΓΚΡΙΘΕΝ ΠΟΣΟ ΤΗΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`
                                ,`ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`
	                            ,COALESCE(`Δείκτης ποσοστού συμμετοχής κοινοπρακτικού δανείου`,'0' ) AS `ΔΕΙΚΤΗΣ ΠΟΣΟΣΤΟΥ ΣΥΜΜΕΤΟΧΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`
                        FROM   syndicatedFlagLoan sfl
                        INNER JOIN syndicatedFlagAmountSum sfa
                        ON     sfa.`ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`=sfl.`ΗΛΕΚΤΡΟΝΙΚΟΣ ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ`
                        LEFT JOIN koinopraktika k
                        ON     k.ExposureKey=sfl.account_number
                        """
)
syndicatedFlagFields.createOrReplaceTempView("syndicatedFlagFields")

Dparty = spark.sql( f"""
                   
                        select distinct CDI,PartyType ,PartyKeyThesis,ROW_NUMBER() OVER ( PARTITION BY CDI    ORDER BY businessdate ASC  ) AS rnk
                        FROM `{env}-cdh`.derivative.dparty p
                        CROSS JOIN param_dates param
                        WHERE p.businessdate=param.eom_lst_wrk_date
                  
                        """
)

Dparty.createOrReplaceTempView("Dparty")

dREcollateral = spark.sql( f"""
                        SELECT DISTINCT  c.* ,param.eom_lst_wrk_date
                        FROM `{env}-cdh`.derivative.dcollateral c
                        CROSS JOIN param_dates param
                        WHERE c.businessdate=param.eom_lst_wrk_date
                        AND   c.collateraltype='RE'
                        """
)

dREcollateral.createOrReplaceTempView("dREcollateral")



PropertyOwnership = spark.sql(f"""
                                SELECT DISTINCT
                                    re.BusinessDate
                                    ,re.OwnershipKey
                                    ,dcol.DCollateralKey
                                    ,dre.RealEstateCepalKey
                                    ,dp.CDI 
                                FROM  `{env}-cdh`.primary.realestateinfo re
                                INNER JOIN dREcollateral dcol
                                ON dcol.RealEstateKey = re.RealEstateKey
                                AND re.OwnerPartyKey = dcol.OwnerPartyKey
                                AND re.BusinessDate = dcol.businessdate
                                LEFT JOIN Dparty dp
                                ON re.OwnerPartyKey = dp.PartyKeyThesis
                                LEFT JOIN `{env}-cdh`.derivative.drealestate dre
                                ON dre.RealEstateKey = re.RealEstateKey
""")

PropertyOwnership.createOrReplaceTempView('PropertyOwnership')

RREinfo = spark.sql( f"""
           SELECT   DISTINCT  
                     e.AgreementKey 
                    ,pop.CDI  
                    ,CASE WHEN 
                            CASE WHEN c.RealEstateCollateralStatusCode='N/A' THEN NULL ELSE c.RealEstateCollateralStatusCode END   IS NULL 
                        OR  CASE WHEN c.RealEstateCollateralStatusCode='N/A' THEN NULL ELSE c.RealEstateCollateralStatusCode 
                            END IN ('Ενεργή') 
                        THEN 'Active' 
                        ELSE 'Inactive'
                    END AS CollateralStatusEng
                    ,p.PartyType 
                    ,ep.partyrelationshiptype 
            FROM dREcollateral c
            INNER JOIN `{env}-cdh`.derivative.drealestate re
            ON c.RealEstateKey = re.RealEstateKey
            AND re.BusinessDate=c.BusinessDate
            INNER JOIN `{env}-cdh`.primary.realestatebuilding reb
            ON reb.RealEstateCepalKey = re.RealEstateCepalKey
            AND reb.businessdate=c.businessdate
            INNER JOIN `{env}-cdh`.derivative.dexposure e 
            ON  e.agreementKey=c.agreementKey
            AND e.businessdate=c.businessdate 
            LEFT JOIN `{env}-cdh`.primary.realestateinfo reinfo
            ON c.RealEstateKey = reinfo.RealEstateKey
            AND reinfo.OwnerPartyKey = c.OwnerPartyKey
            AND reinfo.businessDate=c.businessdate
            INNER JOIN PropertyOwnership po
            ON po.OwnershipKey= reinfo.OwnershipKey
            AND  c.DCollateralKey=po.DCollateralKey
            LEFT JOIN Dparty pop
            ON pop.CDI=po.CDI
            LEFT JOIN Dparty p
            on p.CDI=e.CDI
            LEFT JOIN `{env}-cdh`.primary.exposure    ep
            ON  ep.cdi=po.cdi
            AND ep.agreementKey=c.agreementKey
            AND ep.businessdate=po.businessdate 
            --on p.PartyKeyThesis = REinfo.OwnerPartyKey
            WHERE 1=1
            --AND c.collateraltype='RE'
            AND reb.realestatecategoryCode='Residential'
            
UNION
            SELECT                
                    e.AgreementKey 
                    ,poP.CDI  
                    ,CASE WHEN 
                            CASE WHEN c.RealEstateCollateralStatusCode='N/A' THEN NULL ELSE c.RealEstateCollateralStatusCode END   IS NULL 
                        OR  CASE WHEN c.RealEstateCollateralStatusCode='N/A' THEN NULL ELSE c.RealEstateCollateralStatusCode 
                            END IN ('Ενεργή') 
                        THEN 'Active' 
                        ELSE 'Inactive'
                    END AS CollateralStatusEng
                    ,p.PartyType  
                    ,ep.partyrelationshiptype     
            FROM dREcollateral c
            INNER JOIN `{env}-cdh`.derivative.drealestate re
            ON c.RealEstateKey = re.RealEstateKey
            AND re.BusinessDate=c.BusinessDate
            INNER JOIN `{env}-cdh`.primary.realestatebuilding reb
            ON reb.RealEstateCepalKey = re.RealEstateCepalKey
            AND reb.businessdate=c.businessdate
            INNER JOIN d_closed_perimeter e
            ON  e.agreementKey=c.agreementkey
            LEFT JOIN `{env}-cdh`.primary.realestateinfo reinfo
            ON c.RealEstateKey = reinfo.RealEstateKey
            AND reinfo.OwnerPartyKey = c.OwnerPartyKey
            AND reinfo.businessDate=c.businessdate
            INNER JOIN PropertyOwnership po
            ON po.OwnershipKey= reinfo.OwnershipKey
            AND  c.DCollateralKey=po.DCollateralKey
            LEFT JOIN Dparty pop
            on pop.CDI=po.CDI
            LEFT JOIN Dparty p
            ON p.CDI=e.CDI
            left JOIN closed_perimeter  ep
            ON  ep.cdi=po.cdi
            and ep.agreementKey=c.agreementKey
            WHERE 1=1
           -- and c.collateraltype='RE'
            AND reb.realestatecategoryCode='Residential'
                        """
)
RREinfo.createOrReplaceTempView("RREinfo")

RREpreFinal = spark.sql( f"""
                   SELECT DISTINCT
                        AgreementKey
                        ,CDI
                        ,CASE WHEN PARTYTYPE='Individual' AND PARTYRELATIONSHIPTYPE IN ('Other','Primary Owner'/*,'Co-Owner'*/)  AND CollateralStatusEng='Active' THEN 'Individual Collateral'
                              WHEN (PARTYTYPE='Organization' OR PARTYRELATIONSHIPTYPE IN ('Guarantor'))  AND CollateralStatusEng='Active' THEN 'Organization Collateral'
                        END AS `RRE_ΓΙΑ_ΑΠΦ` 
                    FROM RREinfo
                        """
)
RREpreFinal.createOrReplaceTempView("RREpreFinal")

RREFinal = spark.sql( f"""
                SELECT DISTINCT
                     AgreementKey
                    ,CDI
                    ,`RRE_ΓΙΑ_ΑΠΦ`
                FROM RREpreFinal
                WHERE `RRE_ΓΙΑ_ΑΠΦ` IN('Organization Collateral','Individual Collateral')--OR NOT NULL 
                        """
)
RREFinal.createOrReplaceTempView("RREFinal")

In [0]:

loandata_init = spark.sql( f"""        
                --open accounts
                SELECT DISTINCT
                     p.eomDate
                    ,p.eomWrkDate
                    ,p.CalendarDate
                    ,p.PortfolioId
                    ,p.portfolioName AS PortfolioName
                    ,p.TiresiasName AS Portfolio
                    ,p.TiresiasCode
                    ,SUBSTRING(e.ExposureKey, 4, 100) AS Account_Number
                    ,e.ExposureKey AS Profits_Account_Number
                    ,e.CDI
                    ,e.PartyTaxRegistrationNumber
                    ,REPLACE(REPLACE(SUBSTRING(e.ExposureKey, 4, 100), '_1', ''), '_2', '') AS tiresias_account_number
                    ,CONCAT('bsect-',e.BankingUnitCode) AS product_key
                    ,e.ExposureStatus 
                    ,`{env2}-cdh-delivery`.tiresias.GetTiresiasFile(e.AssetSegment) AS TiresiasFile 
                    ,p.AssignmentDate AS MigrationDate
                    ,`{env2}-cdh-delivery`.tiresias.GetTiresiasCode(e.AssetSegment,e.Mechanism) AS Tiresias_Code_Init
                    ,e.Mechanism AS acc_mechanism
                    ,e.agreementKey AS agreement_number
                    ,e.openDate AS origination_date
                    ,e.maturityDate 
                    ,CASE 
                        WHEN COALESCE(e.originationamount, 0) < 1 THEN e.InitialAssignmentAmount--e.DueAmount
                        ELSE e.originationamount 
                    END AS origination_amount
                    ,e.isactiveflag
                    ,e.ClosedDate
                    ,CASE WHEN e.InstalmentFrequency NOT IN (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 18, 24, 30, 36) THEN 0 
                        ELSE e.InstalmentFrequency 
                    END AS InstalmentFrequency
                    ,i.NumberOfInstallments
                    ,ABS(e.NextInstallmentAmount) AS InstallmentAmount 
                    ,CASE WHEN e.ExposureStatus <>'Normal' AND e.DaysPastDueFinal=0 THEN e.DaysPastDuePrincipal 
                    ELSE e.DaysPastDueFinal
                    END AS DPD
                    ,CASE WHEN e.ExposureStatus IN ('Define Delay','Writen Off')  THEN 0
                        WHEN e.ExposureStatus='Normal' THEN OutstandingTotalBalance
                        ELSE COALESCE(e.Principal,0) + COALESCE(e.Interest,0) + COALESCE(e.UnrealizedInterest,0) + COALESCE(e.Expense,0) + COALESCE(e.Commission,0) +COALESCE(e.Accrued,0)
                    END AS NORMAL_BALANCE
                    ,CASE WHEN e.ExposureStatus IN ('Define Delay','Writen Off')  THEN OutstandingTotalBalance
                        WHEN e.ExposureStatus='Normal' THEN 0
                        ELSE COALESCE(e.DuePrincipal,0) + COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.Penaltyinterest,0) + COALESCE(e.PenaltyUnrealizedinterest,0) + COALESCE(e.DueExpense,0) + COALESCE(e.DueCommission,0) + COALESCE(e.DueAccrued,0)
                    END AS  OVERDUE_BALANCE
                    ,law.UnderLawType   
                    ,e.PEERestructureTypeCode AS pee 
                    , CASE 
                        WHEN COALESCE(
                                CAST(settle.implementationDate AS DATE),
                                rest.ImplementationDate,
                                e.restructuringPeeDate,
                                e.lastRestructuringDate
                            ) IN (DATE('1900-01-01'), DATE('1753-01-01')) THEN NULL
                        ELSE COALESCE(
                                CAST(settle.implementationDate AS DATE),
                                rest.ImplementationDate,
                                e.restructuringPeeDate,
                                e.lastRestructuringDate
                            )
                    END AS restructuredate
                    /*,settle.implementationDate AS implementationDate_set
                    ,rest.ImplementationDate AS ImplementationDate_Res
                    , e.restructuringPeeDate
                    , e.lastRestructuringDate*/
                     ,e.HasActiveSettlementFlag
                     ,COALESCE(settle.SettlementKey,fallback.SettlementKey) AS settlementCode
                    ,COALESCE(settle.SettlementStatusCode,fallback.SettlementStatusCode) AS settlementStatus
                    ,COALESCE(CAST(settle.ExpiredDate AS DATE) ,fallback.ExpiredDate)AS settlementExpiryDate
                    ,COALESCE(settle.OverdueDate ,fallback.OverdueDate)AS settlementOverdueDate
                    ,COALESCE(CAST(settle.implementationDate AS DATE) ,fallback.implementationDate)AS settlementImplementationDate
                    ,pay.last_payment_date
                    ,COALESCE(pay.total_payment_amount,0) AS total_payment_amount
                    ,COALESCE(pay.prv_month_payment_amount,0) AS prv_month_payment_amount
                    ,CASE WHEN e.AssetSegment='ML' THEN TRUE  ELSE FALSE  END AS IS_HOUSING 
                    ,e.AssetSegment 
                    ,e.productCode AS prft_product_code 
                    ------------------new
                    ,CASE WHEN ggf.exposurekey IS NOT NULL THEN '0' ELSE '' END AS `ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`
                    ,CASE WHEN sfl.account_number IS NOT NULL THEN `ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ` ELSE '' 
                     END AS `ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
                    ,CASE WHEN sfl.account_number IS NOT NULL THEN CAST(`ΣΥΝΟΛΙΚΟ ΕΓΚΡΙΘΕΝ ΠΟΣΟ ΤΗΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`AS string) ELSE ''  
                     END  AS `ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
                    ,CASE WHEN sfl.account_number IS NOT NULL THEN CAST(`ΔΕΙΚΤΗΣ ΠΟΣΟΣΤΟΥ ΣΥΜΜΕΤΟΧΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`AS string) ELSE ''
                      END AS `ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
                    ,CASE WHEN rre.`RRE_ΓΙΑ_ΑΠΦ`='Individual Collateral' THEN '2' 
                         WHEN rre.`RRE_ΓΙΑ_ΑΠΦ`='Organization Collateral' THEN ''  
                         ELSE '1' 
                     END AS `RRE_ΓΙΑ_ΑΠΦ` 
                FROM d_exposure_perimeter e 
                INNER JOIN portfolio p
                ON    e.portfolioname=p.PortfolioName and e.businessdate=eomWrkDate---effective?? business date within the month dates??--DIFF for middle of month run
                LEFT JOIN installments i
                ON i.Exposurekey=e.Exposurekey
                LEFT JOIN payment2 pay 
                ON  pay.normalized_loanbankcode=e.ExposureKey
                LEFT JOIN underlaw law 
                ON  law.cdi=e.CDI
                LEFT JOIN `{env}-cdh`.primary.SettlementExposure settle
                ON  settle.exposurekey=e.ExposureKey
                AND (settle.CDI = e.CDI OR settle.CDI IS NULL)
                AND e.HasActiveSettlementFlag=TRUE
                AND settle.businessdate=e.businessdate
                LEFT JOIN settlement_ranked fallback
                ON fallback.exposurekey = e.ExposureKey
                AND fallback.businessdate = e.businessdate
                AND fallback.rn = 1  -- only one fallback row per exposurekey/businessdate
                AND e.HasActiveSettlementFlag = TRUE
                AND NOT (fallback.CDI = e.CDI OR fallback.CDI IS NULL) 
                LEFT JOIN restructuring_2 rest
                ON e.ExposureKey = rest.OldAccountNumber
                LEFT JOIN GovernmentGuaranteeFlag ggf
                ON ggf.exposurekey=e.ExposureKey
                LEFT JOIN syndicatedFlagFields sfl
                ON sfl.account_number=e.ExposureKey
                LEFT JOIN RREfinal rre
                ON rre.AgreementKey=e.AgreementKey
                AND rre.CDI=e.CDI

                WHERE 1=1
                AND e.openDate<=p.eomWrkDate
                and e.isactiveflag = TRUE -- SINCE THE closed perimeter is on the rest of the union ,to avoid skipping the accounts that opened on the last day of month not being included
                AND (e.closedDate  IS NULL OR e.closedDate BETWEEN p.somDate AND p.eomWrkDate)
                AND e.openDate !=COALESCE(e.closedDate,'9999-12-31') --exclude loans that open and closed same date--error form ops
                AND NOT EXISTS (SELECT 1 FROM  l1_l2_sub sub WHERE sub.DEBIT_OPEN_ACCOUNT=e.ExposureKey)  --l1-l2 subsidy p-cdh.landed.profits_lns_ext_subsidy_hd  //landing
                
       UNION

            SELECT DISTINCT
            --closed_Accounts
                 p.eomDate
                ,p.eomWrkDate
                ,p.CalendarDate
                ,p.PortfolioId
                ,p.portfolioName
                ,p.TiresiasName AS Portfolio
                ,p.TiresiasCode
                ,SUBSTRING(e.ExposureKey, 4, 100) AS Account_Number
                ,e.ExposureKey AS Profits_Account_Number
                ,e.CDI
                ,e.PartyTaxRegistrationNumber
                ,REPLACE(REPLACE(SUBSTRING(e.ExposureKey, 4, 100), '_1', ''), '_2', '') AS tiresias_account_number
                ,CONCAT('bsect-',e.BankingUnitCode) AS product_key
                ,e.ExposureStatus 
                ,`{env2}-cdh-delivery`.tiresias.GetTiresiasFile(e.AssetSegment) AS TiresiasFile 
                ,p.AssignmentDate AS MigrationDate
                ,`{env2}-cdh-delivery`.tiresias.GetTiresiasCode(e.AssetSegment,e.Mechanism) AS Tiresias_Code_Init
                ,e.Mechanism AS acc_mechanism
                ,e.agreementKey AS agreement_number
                ,e.openDate AS origination_date
                ,e.maturityDate 
                ,CASE 
                    WHEN COALESCE(e.originationamount, 0) < 1 THEN e.InitialAssignmentAmount--e.DueAmount
                    ELSE e.originationamount 
                END AS origination_amount
                ,e.isactiveflag
                ,e.ClosedDate
                ,CASE WHEN e.InstalmentFrequency NOT IN (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 18, 24, 30, 36) THEN 0 
                    ELSE e.InstalmentFrequency 
                END AS InstalmentFrequency
                ,i.NumberOfInstallments
                ,ABS(e.NextInstallmentAmount) AS InstallmentAmount 
                ,CASE WHEN e.ExposureStatus <>'Normal' AND e.DaysPastDueFinal=0 THEN e.DaysPastDuePrincipal 
                    ELSE e.DaysPastDueFinal
                END AS DPD
                ,CASE WHEN e.ExposureStatus IN ('Define Delay','Writen Off')  THEN 0
                    WHEN e.ExposureStatus='Normal' THEN OutstandingTotalBalance
                    ELSE COALESCE(e.Principal,0) + COALESCE(e.Interest,0) + COALESCE(e.UnrealizedInterest,0) + COALESCE(e.Expense,0) + COALESCE(e.Commission,0) +COALESCE(e.Accrued,0)
                END AS NORMAL_BALANCE
                ,CASE WHEN e.ExposureStatus IN ('Define Delay','Writen Off')  THEN OutstandingTotalBalance
                    WHEN e.ExposureStatus='Normal' THEN 0
                    ELSE COALESCE(e.DuePrincipal,0) + COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.Penaltyinterest,0) + COALESCE(e.PenaltyUnrealizedinterest,0) + COALESCE(e.DueExpense,0) + COALESCE(e.DueCommission,0) + COALESCE(e.DueAccrued,0)
                END AS  OVERDUE_BALANCE
                ,law.UnderLawType  
                ,e.PEERestructureTypeCode AS pee
               , CASE 
                        WHEN COALESCE(
                                CAST(settle.implementationDate AS DATE),
                                rest.ImplementationDate,
                                e.restructuringPeeDate,
                                e.lastRestructuringDate
                            ) IN (DATE('1900-01-01'), DATE('1753-01-01')) THEN NULL
                        ELSE COALESCE(
                                CAST(settle.implementationDate AS DATE),
                                rest.ImplementationDate,
                                e.restructuringPeeDate,
                                e.lastRestructuringDate
                            )
                    END AS restructuredate

                /*,settle.implementationDate AS implementationDate_set
                ,rest.ImplementationDate AS ImplementationDate_Res
                , e.restructuringPeeDate
                , e.lastRestructuringDate */
                ,e.HasActiveSettlementFlag
                ,COALESCE(settle.SettlementKey,fallback.SettlementKey) AS settlementCode
                ,COALESCE(settle.SettlementStatusCode,fallback.SettlementStatusCode) AS settlementStatus
                ,COALESCE(CAST(settle.ExpiredDate AS DATE) ,fallback.ExpiredDate)AS settlementExpiryDate
                ,COALESCE(settle.OverdueDate ,fallback.OverdueDate)AS settlementOverdueDate
                ,COALESCE(CAST(settle.implementationDate AS DATE) ,fallback.implementationDate)AS settlementImplementationDate 
                ,pay.last_payment_date
                ,COALESCE(pay.total_payment_amount,0) AS total_payment_amount
                ,COALESCE(pay.prv_month_payment_amount,0) AS prv_month_payment_amount
                ,CASE WHEN e.AssetSegment='ML' THEN TRUE  ELSE FALSE  END AS IS_HOUSING 
                ,e.AssetSegment
                ,e.productCode as prft_product_code 
                ------------------new
                ,CASE WHEN ggf.exposurekey IS NOT NULL THEN '0' ELSE '' END AS `ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`
                ,CASE WHEN sfl.account_number IS NOT NULL THEN `ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ` ELSE '' 
                 END AS `ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
                ,CASE WHEN sfl.account_number IS NOT NULL THEN CAST(`ΣΥΝΟΛΙΚΟ ΕΓΚΡΙΘΕΝ ΠΟΣΟ ΤΗΣ ΣΥΜΒΑΣΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`AS string) ELSE ''  
                 END  AS `ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
                ,CASE WHEN sfl.account_number IS NOT NULL THEN CAST(`ΔΕΙΚΤΗΣ ΠΟΣΟΣΤΟΥ ΣΥΜΜΕΤΟΧΗΣ ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ ΔΑΝΕΙΟΥ`AS string) ELSE '' 
                      END AS `ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
                ,CASE WHEN rre.`RRE_ΓΙΑ_ΑΠΦ`='Individual Collateral' THEN '2' 
                      WHEN rre.`RRE_ΓΙΑ_ΑΠΦ`='Organization Collateral' THEN '' 
                      ELSE '1'
                 END AS `RRE_ΓΙΑ_ΑΠΦ` 

            FROM  d_closed_perimeter e 
            INNER JOIN portfolio p
            ON    e.portfolioname=p.PortfolioName --and e.businessdate=eomWrkDate---effective?? business date within the month dates??--DIFF for middle of month run
            LEFT JOIN installments i
            ON i.Exposurekey=e.Exposurekey
            LEFT JOIN payment2 pay 
            ON  pay.normalized_loanbankcode=e.ExposureKey 
            LEFT JOIN underlaw law 
            ON  law.cdi=e.CDI
            LEFT JOIN `{env}-cdh`.primary.SettlementExposure settle
            ON  settle.exposurekey=e.ExposureKey
            AND (settle.CDI = e.CDI OR settle.CDI IS NULL)
            AND settle.businessdate=e.businessdate
            AND e.HasActiveSettlementFlag=TRUE
            LEFT JOIN settlement_ranked fallback
            ON fallback.exposurekey = e.ExposureKey
            AND fallback.businessdate = e.businessdate
            AND fallback.rn = 1  -- only one fallback row per exposurekey/businessdate
            AND e.HasActiveSettlementFlag = TRUE
            AND NOT (fallback.CDI = e.CDI OR fallback.CDI IS NULL) 
            LEFT JOIN restructuring_2 rest
            ON e.ExposureKey = rest.OldAccountNumber
            LEFT JOIN GovernmentGuaranteeFlag ggf
            ON ggf.exposurekey=e.ExposureKey
            LEFT JOIN syndicatedFlagFields sfl
            ON sfl.account_number=e.ExposureKey
            LEFT JOIN RREfinal rre 
            ON rre.AgreementKey=e.AgreementKey 
            AND rre.CDI=e.CDI 
       
            WHERE 1=1
            AND e.openDate<=p.eomWrkDate
            AND (e.closedDate  IS NULL OR e.closedDate BETWEEN p.somDate AND p.eomWrkDate)
            AND e.openDate != e.businessdate --  AND e.openDate !=COALESCE(e.closedDate,'9999-12-31')--exclude loans that oepn and closed same date--error form ops!! THAT WONT WORK HERE
            AND NOT EXISTS (SELECT 1 FROM  l1_l2_sub sub WHERE sub.DEBIT_OPEN_ACCOUNT=e.ExposureKey)  --l1-l2 subsidy p-cdh.landed.profits_lns_ext_subsidy_hd  //landing
                          
                        """
)

loandata_init.createOrReplaceTempView("loandata_init")

#display(loandata_init)

In [0]:
rem_install = spark.sql(f"""
                        SELECT  l.account_number,count(1) as remaining_installments
                        FROM   loandata_init l
                        INNER JOIN `{env}-cdh`.primary.paymentschedule ps
                        ON  l.profits_account_number=ps.exposurekey
                        AND ps.businessdate=l.eomWrkDate
                        WHERE HasActiveSettlementFlag IS FALSE
                        AND ExposureStatus in('Normal','Overdue')
                        AND COALESCE(NumberOfInstallments,0)>0
                        AND isactiveflag IS TRUE
                        AND ps.PaymentScheduleType='Loan Payment Schedule'
                        AND ps.InstallmentType='Installment'
                        AND CAST(ps.InstallmentDate AS DATE) > l.eomWrkDate
                        AND l.maturityDate > l.eomWrkDate
                        GROUP BY  l.account_number
            
                        """
)
rem_install.createOrReplaceTempView("rem_install")
#-----------------------------------------------------------------------
#1st update , get remaining installments and master-slave flag and connection
loandata_upd1 = spark.sql(f"""
                        SELECT  DISTINCT
                               l.*
                               ,ri.remaining_installments
                               ,CASE WHEN s.MasterAccount IS NOT NULL THEN TRUE
                                     ELSE FALSE END AS is_slave
                               ,CASE WHEN s.MasterAccount IS NOT NULL THEN s.MasterAccount
                                     ELSE NULL END AS MasterAccount
                              
                        FROM   loandata_init l
                        LEFT JOIN rem_install ri 
                        ON ri.account_number=l.account_number
                        LEFT JOIN master_slave_Accounts_ranked s
                        ON   s.slaveAccount=l.Profits_Account_Number 
                        AND rn =1
            
                        """
)
loandata_upd1.createOrReplaceTempView("loandata_upd1")

#-----------------------------------------------------------------------
# Implementation calculations
#-----------------------------------------------------------------------
# CALCULATE the implementations/modifications
loandata_upd2 = spark.sql(f"""
                        SELECT  DISTINCT
                                l.*
                               ,CASE WHEN l.HasActiveSettlementFlag IS TRUE then 'settlement'
                                     WHEN woof.slaveaccount IS NOT NULL THEN 'CL-woff'
                                     WHEN l.is_slave IS TRUE THEN 'CL-slv'
                                     WHEN l.pee IS NOT NULL THEN 'CL-mod'
                                     ELSE NULL
                                END AS impl_type
                                ,CASE WHEN l.HasActiveSettlementFlag IS TRUE then '17'
                                     
                                      WHEN l.is_slave IS TRUE THEN NULL
                                      WHEN l.pee IS NOT NULL THEN l.pee
                                     ELSE NULL
                                END AS impl_pee
                        FROM   loandata_upd1 l
                        LEFT JOIN master_slave_Accounts woof on  woof.slaveAccount=l.Profits_Account_Number 
                        AND( EntityRelationshipType='Restructured Loan - Writeoff Rltnp'  OR EntityRelationshipSubType='WDN')
            
                        """
)
loandata_upd2.createOrReplaceTempView("loandata_upd2")

In [0]:
#create table form dataframe 
#because of too many fields and records the updated below were taking too long

# Check if the table exists and drop it if it does
if spark.catalog.tableExists(f"`{env2}-cdh-delivery`.tiresias.loan_data"):
   spark.sql(f"DROP TABLE `{env2}-cdh-delivery`.tiresias.loan_data")

# Create and fill the table with the DataFrame's info
loandata_upd2.write.format("delta").mode("overwrite").saveAsTable(f"`{env2}-cdh-delivery`.tiresias.loan_data")


In [0]:
#raise error if no product copde found
check_1 = spark.sql(f"""
                        SELECT COUNT(*) as cnt
                        FROM   `{env2}-cdh-delivery`.tiresias.loan_data l
                        WHERE l.Tiresias_Code_Init IS NULL OR l.TiresiasFile IS NULL
            
                        """
)
#check_1.createOrReplaceTempView("check_1")
#collect results
count = check_1.collect()[0]['cnt']
if count > 10:
    raise Exception('Tiresias product code missing for more than 10 records')


In [0]:
# Raise error for double loans
check_dup = spark.sql(f"""
    SELECT profits_account_number,
           COUNT(*) as cnt,
           COUNT(DISTINCT isactiveflag) as flag_variation
    FROM `{env2}-cdh-delivery`.tiresias.loan_data l
    GROUP BY profits_account_number 
    HAVING COUNT(*) > 1
""")

diff_flag_duplicates = check_dup.filter("flag_variation > 1")  # Get the ones that have diff in flag
diff_flag_list = [row['profits_account_number'] for row in diff_flag_duplicates.collect()]

if diff_flag_list:
    # Delete rows with flag = false for these profits_account_numbers
    profits_numbers_str = ",".join([f"'{x}'" for x in diff_flag_list])
    spark.sql(f"""
        DELETE FROM `{env2}-cdh-delivery`.tiresias.loan_data
        WHERE profits_account_number IN ({profits_numbers_str}) AND isactiveflag = FALSE
    """)

# Recheck duplicates after deletion
remaining_dupes = spark.sql(f"""
    SELECT profits_account_number, COUNT(*) as cnt
    FROM `{env2}-cdh-delivery`.tiresias.loan_data
    GROUP BY profits_account_number
    HAVING COUNT(*) > 1
""")

# Collect results
remaining_dupes_list = remaining_dupes.collect()
if remaining_dupes_list and remaining_dupes_list[0]['cnt'] > 0:
    raise Exception('Double Loan Found!')

In [0]:
#--new updates loandate--KEEP THESE
#-----------------------------------------------------------------------
#------FMst1
master_slave = spark.sql(f"""
                        SELECT  DISTINCT
                              slv.Profits_Account_Number
                              ,slv.MasterAccount 
                              ,slv.cdi
                              , CASE WHEN mst.impl_pee IS NULL THEN NULL 
                                     ELSE slv.impl_type
                               END AS impl_type
                              , CASE WHEN  mst.impl_pee IS NULL THEN NULL
                                     ELSE  mst.impl_pee 
                              END AS impl_pee
                              , mst.DPD
                              , mst.ExposureStatus
                              , mst.Tiresias_Code_Init
                              , mst.TiresiasFile
                               
                        FROM   `{env2}-cdh-delivery`.tiresias.loan_data slv
                        LEFT JOIN `{env2}-cdh-delivery`.tiresias.loan_data mst 
                        ON  mst.Profits_Account_Number=slv.MasterAccount 
                        WHERE slv.is_slave IS TRUE
            
                        """
)
master_slave.createOrReplaceTempView("master_slave")
#---------------------------
#MERGE
merge_tiresias_loan_data_1 = spark.sql(f"""
            MERGE INTO  `{env2}-cdh-delivery`.tiresias.loan_data AS t 
            USING (
                SELECT *
                FROM master_slave 
            ) s 
            ON (s.Profits_Account_Number = t.Profits_Account_Number)
            WHEN MATCHED THEN
            UPDATE SET
                      t.impl_type=s.impl_type
                    , t.impl_pee=s.impl_pee
                    , t.DPD=s.DPD
                    , t.ExposureStatus=s.ExposureStatus
                    , t.Tiresias_Code_Init=s.Tiresias_Code_Init
                    ,t.TiresiasFile=s.TiresiasFile
                
""")
#display(merge_tiresias_loan_data_1)  


In [0]:
#-----------------------------------------------------------------------
#------simple woff no slave flag !!!
w_off = spark.sql(f"""
                        SELECT DISTINCT woff.Profits_Account_Number, woff.account_number, agr.impl_pee  
                        FROM `{env2}-cdh-delivery`.tiresias.loan_data woff
                        JOIN (
                        SELECT agreement_number, impl_type, impl_pee, RANK() OVER(PARTITION BY  agreement_number ORDER BY  account_number) rnk
                        FROM `{env2}-cdh-delivery`.tiresias.loan_data
                        WHERE impl_type='CL-mod'
                        ) agr 
                        ON agr.agreement_number=woff.agreement_number
                        AND agr.rnk=1
                        WHERE woff.impl_type='CL-woff'
                        AND woff.is_slave IS FALSE
            
                        """
)
w_off.createOrReplaceTempView("w_off")

#---------------------------
#MERGE
merge_tiresias_loan_data_2 = spark.sql(f"""
            MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_data AS t 
            USING (
                SELECT *
                FROM w_off 
            ) s 
            ON (s.Profits_Account_Number = t.Profits_Account_Number)
            WHEN MATCHED THEN
            UPDATE SET  t.impl_pee=s.impl_pee
                    
                
""")
#display(merge_tiresias_loan_data_2) 


In [0]:
#--for the nulls set 10 as pee
merge_tiresias_loan_data_3 = spark.sql(f"""
            MERGE INTO`{env2}-cdh-delivery`.tiresias.loan_data AS t 
            USING (
                SELECT *
                FROM `{env2}-cdh-delivery`.tiresias.loan_data
                WHERE isactiveflag IS TRUE
                AND   impl_type='CL-woff'
                AND   impl_pee IS NULL
            ) s 
            ON (s.Profits_Account_Number = t.Profits_Account_Number)
            WHEN MATCHED THEN
            UPDATE SET  t.impl_pee='10'
                    
                
""")
#display(merge_tiresias_loan_data_3) 


In [0]:
#---------------------------------------------------------------
#----cl-closed
cl_closed = spark.sql(f"""
                     
                    SELECT DISTINCT 'CL-closed' AS impl_type ,agr.impl_pee 
                            ,agr.agreement_number,cl.Profits_Account_Number
                    FROM `{env2}-cdh-delivery`.tiresias.loan_data cl
                    INNER JOIN(
                        SELECT agreement_number, impl_type, impl_pee, RANK() OVER(PARTITION BY agreement_number ORDER BY impl_type,account_number) rnk
                        FROM `{env2}-cdh-delivery`.tiresias.loan_data
                        WHERE impl_type NOT IN ('settlement')
                        ) agr
                         ON agr.agreement_number=cl.agreement_number
                         AND agr.rnk=1
                    WHERE cl.isactiveflag IS FALSE
            
                        """
)
cl_closed.createOrReplaceTempView("cl_closed")
#---------------------------
#MERGE
merge_tiresias_loan_data_4 = spark.sql(f"""
            MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_data AS t 
            USING (
                SELECT *
                FROM cl_closed 
            ) s 
            ON (s.Profits_Account_Number = t.Profits_Account_Number)
            WHEN MATCHED THEN
            UPDATE SET
                      t.impl_pee=s.impl_pee
                    ,t.impl_type=s.impl_type
                    
                
""")
#display(merge_tiresias_loan_data_4)  


In [0]:
#--merge set Settlement DPD if loan has settlement running
#------settlements
Settlement = spark.sql(f"""
                         SELECT  Profits_Account_Number, DPD, settDPD, settlementCode
                        ,CASE WHEN settDPD=0 THEN 0 
                            ELSE  ov_bal 
                        END AS ov_bal
                        ,CASE WHEN settDPD=0 THEN 'Normal' 
                            ELSE  'Overdue' 
                        END AS loan_status
                        ,CASE WHEN settDPD=0 AND nrm_bal=0 THEN ov_bal 
                            ELSE nrm_bal 
                        END AS nrm_bal
                        ,ov_bal  AS xxx_ov_bal
                        ,nrm_bal AS xxx_nrm_bal
                        ,SettlementStatus
                        FROM  (
                        SELECT Profits_Account_Number, DPD,  SettlementStatus,   settlementOverdueDate
                             ,COALESCE(DATEDIFF(p.eomDate, settlementOverdueDate), 0) AS settDPD
                             ,settlementCode
                             , COALESCE(OVERDUE_BALANCE,0) ov_bal
                             , COALESCE(NORMAL_BALANCE,0) nrm_bal
                             , COALESCE(OVERDUE_BALANCE,0)+COALESCE(NORMAL_BALANCE,0) tot_bal
                        FROM  `{env2}-cdh-delivery`.tiresias.loan_data l
                        INNER JOIN portfolio p ON p.PortfolioName=l.PortfolioName
                        WHERE l.impl_type='settlement'
                        AND   P.processSettlementFlag IS TRUE
                        ) as d
            
                        """
)
Settlement.createOrReplaceTempView("Settlement")

#---------------------------
#MERGE
merge_tiresias_loan_data_5 = spark.sql(f"""
            MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_data as t 
            USING (
                SELECT *
                FROM Settlement 
            ) s 
            ON (s.Profits_Account_Number = t.Profits_Account_Number)
            WHEN MATCHED THEN
            UPDATE SET
                     t.DPD=s.settDPD
                    ,t.OVERDUE_BALANCE=s.ov_bal
                    ,t.NORMAL_BALANCE=s.nrm_bal
                    ,t.ExposureStatus=s.loan_status
                    
                
""")
#display(merge_tiresias_loan_data_5)


In [0]:
#--fixes for slave dpd=0 and ov_bal>0
merge_tiresias_loan_data_6 = spark.sql(f"""
            MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_data AS t 
            USING(
                SELECT *
                FROM  `{env2}-cdh-delivery`.tiresias.loan_data
                WHERE isactiveflag IS TRUE
                AND   is_slave IS TRUE
                AND   DPD=0
                AND   OVERDUE_BALANCE >0
            ) s 
            ON (s.Profits_Account_Number = t.Profits_Account_Number)
            WHEN MATCHED THEN
            UPDATE SET  t.OVERDUE_BALANCE=0 
                   , t.NORMAL_BALANCE=s.OVERDUE_BALANCE + s.NORMAL_BALANCE
                
""")
#display(merge_tiresias_loan_data_3)  

In [0]:
check_dup = spark.sql(f"""
    SELECT profits_account_number,
           COUNT(*) as cnt
    FROM `{env2}-cdh-delivery`.tiresias.loan_data
    GROUP BY profits_account_number 
    HAVING COUNT(*) > 1
""")

dup_list = check_dup.collect()

if any(row['cnt'] > 1 for row in dup_list):
    raise Exception('Double Loan Found!')
else:
    print("No duplicates found.")

In [0]:
#customer_loan_data
loancustomerdata = spark.sql(f"""
                             SELECT DISTINCT  
                             l.eomdate
                            ,l.portfolioID
                            ,l.portfolioname
                            ,l.Portfolio
                            ,l.Account_Number
                            ,e.CDI
                            ,e.Partytaxregistrationnumber
                            ,e.partyrelationshiptype
                            ,CASE WHEN LEFT(NationalNumber,1)  IN ('&','/''X','C','M','A')  OR (LEFT(NationalNumber, 1) RLIKE '^[0-9]$') THEN NULL
                                    ELSE p.NationalNumber
                            END as NationalNumber
                            ,CASE WHEN LEFT(NationalNumber,1)  IN ('&','/''X','C','M','A') OR (LEFT(NationalNumber, 1) RLIKE '^[0-9]$') THEN NULL
                                    ELSE p.NationalNumberIssueDate
                            END as NationalNumberIssueDate
                            ,CASE WHEN LEFT(NationalNumber,1)  IN ('&','/''X','C','M','A')  OR (LEFT(NationalNumber, 1) RLIKE '^[0-9]$') THEN FALSE
                                    ELSE TRUE  END AS national_id_flg
                    FROM `{env2}-cdh-delivery`.tiresias.loan_data l 
                    INNER JOIN `{env}-cdh`.primary.exposure e
                    ON  l.Profits_Account_Number=e.exposurekey
                    AND e.sourcesystem='profits'
                    AND e.businessdate=l.eomWrkDate
                    INNER JOIN  `{env}-cdh`.primary.party p
                    ON p.CDI=e.CDI 
                    AND p.businessdate=e.businessdate
                    AND p.sourcesystem='profits'
                    UNION
                    SELECT DISTINCT  
                             l.eomdate
                            ,l.portfolioID
                            ,l.portfolioname
                            ,l.Portfolio
                            ,l.Account_Number
                            ,e.CDI
                            ,e.Partytaxregistrationnumber
                            ,e.partyrelationshiptype
                            ,CASE WHEN LEFT(NationalNumber,1)  IN ('&','/''X','C','M','A')  OR (LEFT(NationalNumber, 1) RLIKE '^[0-9]$') THEN NULL
                                    ELSE p.NationalNumber
                            END as NationalNumber
                            ,CASE WHEN LEFT(NationalNumber,1)  IN ('&','/''X','C','M','A') OR (LEFT(NationalNumber, 1) RLIKE '^[0-9]$') THEN NULL
                                    ELSE p.NationalNumberIssueDate
                            END as NationalNumberIssueDate
                            ,CASE WHEN LEFT(NationalNumber,1)  IN ('&','/''X','C','M','A')  OR (LEFT(NationalNumber, 1) RLIKE '^[0-9]$') THEN FALSE
                                    ELSE TRUE  END AS national_id_flg
                            
                    FROM `{env2}-cdh-delivery`.tiresias.loan_data l
                    INNER JOIN closed_perimeter e
                    ON  l.Profits_Account_Number=e.exposurekey 
                    INNER JOIN  `{env}-cdh`.primary.party p
                    ON p.CDI=e.CDI 
                    AND p.businessdate=e.businessdate
                    AND p.sourcesystem='profits'
                    """
) 
loancustomerdata.createOrReplaceTempView("loancustomerdata")


In [0]:
# tiresias.customerdata
customerdata = spark.sql(f"""
                            SELECT DISTINCT 
                            port.portfolioid
                            ,port.portfolioname
                            ,p.CDI
                            ,p.TaxRegistrationNumber
                            ,CASE WHEN LEFT(p.NationalNumber,1)  IN ('&','/''X','C','M','A')  OR (LEFT(p.NationalNumber, 1) RLIKE '^[0-9]$') THEN NULL
                                    ELSE p.NationalNumber
                            END as NationalNumber
                            ,CASE WHEN LEFT(p.NationalNumber,1)  IN ('&','/''X','C','M','A') OR (LEFT(p.NationalNumber, 1) RLIKE '^[0-9]$') THEN NULL
                                    ELSE p.NationalNumberIssueDate
                            END as NationalNumberIssueDate 
                            ,CASE WHEN p.PartyType='Organization' THEN 2
                                    WHEN p.PartyType='Individual'   THEN 1
                                    END
                            AS cust_type
                            ,CASE WHEN p.PartyType='Individual' THEN  GenderCode
                                    ELSE NULL END 
                            AS GenderCode
                            ,CASE WHEN p.PartyType='Individual' THEN  FirstNameGR
                                    ELSE NULL END 
                            AS FirstName 
                            ,CASE WHEN p.PartyType='Individual' THEN  LastNameGR
                                    ELSE NULL END
                            AS LastName
                            ,CASE WHEN p.PartyType='Individual' THEN  FirstFatherNameGR
                                    ELSE NULL END
                            AS FirstFatherName
                            ,CASE WHEN p.PartyType='Organization' THEN  LastNameGR
                                    ELSE NULL END
                            AS CompanyName 
                            ,LEFT(CONCAT(COALESCE(CAST(p.EconomicActivityCODE AS STRING),''),'00000000'),8) AS ccode
                            ,CASE WHEN p.PartyType='Organization' THEN LegalForm
                                    ELSE NULL END
                            AS LegalForm
                            ,CASE WHEN p.PartyType='Organization' THEN 
                                        CASE WHEN LegalForm='ΙΔΙΩΤΕΣ - ΑΤΟΜΙΚΕΣ ΕΠΙΧΕΙΡΗΣΕΙΣ'  THEN TRUE ELSE FALSE END
                                    ELSE FALSE END 
                            AS ispersonalcompany
                            ,port.eomdate
                            ,p.BUSINESSDATE
                        FROM `{env}-cdh`.primary.party p
                        INNER JOIN portfolio port
                        ON port.portfolioname=p.portfolioname
                        INNER JOIN loancustomerdata lcd --REPLACING EXISTS IN
                        ON lcd.cdi=p.cdi
                        WHERE p.businessdate=port.eomWrkDate
                        AND p.sourcesystem='profits'
                        /*AND NOT EXISTS (
                           SELECT 1 
                           FROM `{env2}-cdh-delivery`.tiresias.customer cust 
                           WHERE cust.CDI = p.CDI
                       )*/
                        
                        """
)

 
customerdata.createOrReplaceTempView("customerdata")

# customeraddressdata
customeraddressdata = spark.sql(f"""
                            SELECT DISTINCT
                             cd.cdi
                            ,cd.ContactDetailType
                            ,cd.ContactDetailTypeCode
                            ,cd.CommunicationFlag
                            ,cd.city
                            ,cd.country
                            ,cd.county
                            ,cd.Street
                            ,cd.streetnumber--NOT FILLED(CHECK IN WHOLE TABLE)
                            ,cd.postalcode
                    FROM `{env}-cdh`.primary.contactdetails cd
                    CROSS JOIN param_dates param
                    inner join customerdata CUST 
                    on CUST.CDI=CD.CDI
                    WHERE cd.businessdate=param.eom_lst_wrk_date
                    AND cd.sourcesystem='profits'
                   
                        """
)

customeraddressdata.createOrReplaceTempView("customeraddressdata")


In [0]:

INIT_before_update=spark.sql(f"""
                                UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer 
								SET has_error = FALSE, errors = NULL
								WHERE tiresias_export_flg = 1
								AND has_error = TRUE
        """)


#display(INIT_before_update)

In [0]:
get_error_from_BUSINESS_hist =spark.sql(f"""	
                                        MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_customer t
										USING (
										SELECT DISTINCT 
												--cl.sk_id,
												cl.account_number,cl.cdi,
												h.rpt_date Tiresias_last_export_dt
            									,CASE   WHEN h.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IN ('1', '9', '5')
                     										 	OR  `ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΧΡΗΣΗΣ_ΟΡΙΟΥ_ΧΟΡΗΓΗΣΕΩΝ_ΑΝΑΚΥΚΛΟΥΜΕΝΗΣ_ΜΟΡΦΗΣ` = '1' 
                                							THEN TRUE 
													    ELSE FALSE
                 								 END send_as_closed_relation
                          						, COALESCE(h.has_error, false) has_error
                                				, h.errors
											FROM `{env2}-cdh-delivery`.tiresias.loan_customer cl
											INNER JOIN `{env2}-cdh-delivery`.tiresias.export_business_hist h  --h.cl_sk_id = cl.sk_id
											ON h.cdi = cl.cdi 
           									AND cl.account_number = h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
                    						and cl.portfolioname=h.portfolio--added for dupl
											INNER JOIN param_dates pp
           									ON pp.previous_eom_date = h.rpt_date
											WHERE cl.Tiresias_File = 'B'
           									AND cl.Tiresias_export_flg = 1
											 
											) s
											ON s.cdi = t.cdi and t.account_number = s.account_number
										WHEN MATCHED THEN UPDATE SET 
          								  t.Tiresias_last_export_dt = s.Tiresias_last_export_dt
										, t.send_as_closed_relation = s.send_as_closed_relation
										, t.has_error = s.has_error
										, t.errors = s.errors
										, t.updated = current_date();
		""")


#display(get_error_from_BUSINESS_hist)

In [0]:
get_error_from_CONSUMER_hist=spark.sql(f"""
                                      	MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_customer t
										USING (
										SELECT DISTINCT 
											--cl.sk_id, c
												cl.account_number
            									,cl.cdi
                     							,h.rpt_date Tiresias_last_export_dt
                            					,CASE WHEN h.`ΔΕΙΚΤΗΣ_ΚΛΕΙΣΙΜΑΤΟΣ` IN ('1', '9') 
                                 								OR `ΔΕΙΚΤΗΣ_ΔΙΑΚΟΠΗΣ_ΑΝΟΙΚΤΟΥ_ΔΑΝΕΙΟΥ` = '1' 
                                       					THEN TRUE 
                                            		  ELSE FALSE
                                                END send_as_closed_relation
                                                ,COALESCE(h.has_error, false) has_error
                                                ,h.errors
											FROM `{env2}-cdh-delivery`.tiresias.loan_customer cl
											INNER JOIN `{env2}-cdh-delivery`.tiresias.export_consumer_hist h 
											--ON h.cl_sk_id = cl.sk_id
											ON h.cdi = cl.cdi 
           									AND cl.account_number = h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
                    						and cl.portfolioname=h.portfolio--added for dupl
											INNER JOIN param_dates pp
           									ON pp.previous_eom_date = h.rpt_date
											WHERE cl.Tiresias_File = 'C'
           									AND cl.Tiresias_export_flg = 1
										
											
											) s
										ON s.cdi = t.cdi AND t.account_number = s.account_number 
											--ON (s.sk_id = t.sk_id)
										WHEN MATCHED THEN UPDATE SET
          								  t.Tiresias_last_export_dt = s.Tiresias_last_export_dt
										, t.send_as_closed_relation = s.send_as_closed_relation
										, t.has_error = s.has_error
										, t.errors = s.errors
										, t.updated = current_date()
		""")


#display(get_error_from_CONSUMER_hist)

In [0]:
UPDATE_account_closed=spark.sql(f"""
                                MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_customer t
								USING (
									SELECT a.*
									FROM  `{env2}-cdh-delivery`.tiresias.loan_customer a
									INNER JOIN param_dates pp 
         							ON pp.previous_eom_date = a.Tiresias_last_export_dt
									WHERE Tiresias_export_flg = 1 
         							AND send_as_closed_relation = TRUE 
                					AND has_error = FALSE
                     				AND NU_date IS NULL--maybe remove??!!
									AND NOT EXISTS (
										SELECT 1 
          								FROM `{env2}-cdh-delivery`.tiresias.loan_data  x 
                  						WHERE  x.isactiveflag = TRUE  
                        				AND COALESCE(x.OVERDUE_BALANCE,0) + COALESCE(x.NORMAL_BALANCE,0) > 0    
                            		    AND x.account_number=a.account_number
									) 
								) s
								ON s.cdi = t.cdi  AND t.account_number = s.account_number
								-- ON (s.sk_id = t.sk_id)
								WHEN MATCHED THEN UPDATE SET
        						Tiresias_not_sent_reason = 'account sent as closed', Tiresias_export_flg = 0
  """)

#display(UPDATE_account_closed) 

In [0]:
UPDATE_remove_relation=spark.sql(f"""   UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer 
										SET Tiresias_export_flg=0, Tiresias_not_sent_reason='remove relation' ,send_as_closed_relation = TRUE 
										WHERE remove_relation_dt IS NOT NULL
										AND Tiresias_export_flg = 1 
										AND remove_relation_dt IS NOT NULL
										AND remove_relation_dt < (SELECT eom_date FROM param_dates)
           """)


#display(UPDATE_remove_relation)  

In [0]:
merge__insert_tiresias_customer=spark.sql(f"""MERGE INTO `{env2}-cdh-delivery`.tiresias.customer AS t USING( 
                                          SELECT c.cdi 
                                                ,c.portfolioname
                                                ,c.CUST_TYPE
                                                ,c.TaxRegistrationNumber `AFM`
                                                ,c.NationalNumber	 `ADT`
                                                ,c.NationalNumberIssueDate  `ADT_issue_date`
                                          FROM  customerdata c  ) AS s
                                          ON (s.cdi=t.cdi and t.portfolioName=s.portfolioname)
                                          WHEN NOT MATCHED THEN INSERT ( portfolioname, cdi, cust_type, `AFM`, `ADT`, `ADT_issue_date`)
                                          VALUES (s.portfolioname, s.cdi, s.cust_type, s.`AFM` ,s.`ADT`,s.`ADT_issue_date`)
""")

#display(merge__insert_tiresias_customer) 

In [0]:
merge_tiresias_customer_1 = spark.sql(f"""
                                    MERGE INTO  `{env2}-cdh-delivery`.tiresias.customer AS t 
                                    USING (
                                        SELECT 
                                            t.cdi,
                                            t.portfolioname,
                                            CASE WHEN cc.GenderCode = 'F' THEN 2
                                                 WHEN cc.GenderCode = 'M' THEN 1
                                                 ELSE 3 
                                                 END `ΦΥΛΟ`, -- 3 etairia 1=male 2=female
                                            CASE WHEN cc.CUST_TYPE = 1 THEN `{env2}-cdh-delivery`.tiresias.getUpperGreekLitTiresias(cc.lastName) END `ΕΠΩΝΥΜΟ`, -- only for person
                                            CASE WHEN cc.CUST_TYPE = 1 THEN `{env2}-cdh-delivery`.tiresias.getUpperGreekLitTiresias(cc.firstname) END `ΟΝΟΜΑ`, -- only for person
                                            CASE WHEN cc.CUST_TYPE = 1 THEN `{env2}-cdh-delivery`.tiresias.getUpperGreekLitTiresias(cc.FirstFatherName) END `ΠΑΤΡΩΝΥΜΟ`, -- only for person*/
                                            `{env2}-cdh-delivery`.tiresias.getUpperGreekLitTiresias(ltrim(rtrim(a.street))) `ΟΔΟΣ`, 
                                            a.streetNumber `ΑΡΙΘΜΟΣ`,
                                            `{env2}-cdh-delivery`.tiresias.getUpperGreekLitTiresias(a.city) `ΠΟΛΗ`, 
                                            a.PostalCode `ΤΑΧ. ΚΩΔΙΚΑΣ`,
                                            CASE WHEN cc.CUST_TYPE = 2 THEN `{env2}-cdh-delivery`.tiresias.getUpperGreekLitTiresias(cc.companyname) END `ΕΠΩΝΥΜΙΑ ΕΤΑΙΡΙΑΣ`, 
                                            CASE WHEN cc.CUST_TYPE = 2 THEN cc.ccode END `ΚΛΑΔΟΣ ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`
                                           ,CASE WHEN cc.CUST_TYPE = 2 THEN COALESCE(pm.tiresia_value, 0)  END  `ΝΟΜΙΚΗ ΜΟΡΦΗ ΕΤΑΙΡΕΙΑΣ`
                                        FROM `{env2}-cdh-delivery`.tiresias.customer t
                                        JOIN customerdata cc 
                                        ON cc.cdi = t.cdi 
                                        AND cc.portfolioname = t.portfolioname
                                        LEFT JOIN  `{env2}-cdh-delivery`.tiresias.mapping pm 
                                        ON pm.ptype = 'LEGAL' 
                                        AND pm.ptype_description = cc.LegalForm
                                        LEFT JOIN (
                                            SELECT *, RANK() OVER (PARTITION BY cdi ORDER BY  communicationFlag DESC, ContactDetailTypeCode ) rnk 
                                            FROM  customerAddressData
                                        ) a
                                        ON a.rnk = 1 
                                        AND a.cdi = cc.cdi
                                    ) s 
                                    ON (s.cdi = t.cdi AND t.portfolioname = s.portfolioname)
                                    WHEN MATCHED THEN
                                    UPDATE  SET 
                                            t.Gender = s.`ΦΥΛΟ`,
                                            t.last_name = s.`ΕΠΩΝΥΜΟ`,
                                            t.first_name = s.`ΟΝΟΜΑ`,
                                            t.father_name = s.`ΠΑΤΡΩΝΥΜΟ`,
                                            t.address_name = s.`ΟΔΟΣ`,
                                            t.address_number = s.`ΑΡΙΘΜΟΣ`,
                                            t.city = s.`ΠΟΛΗ`,
                                            t.postal_code = s.`ΤΑΧ. ΚΩΔΙΚΑΣ`,
                                            t.company_name = s.`ΕΠΩΝΥΜΙΑ ΕΤΑΙΡΙΑΣ`,
                                            t.`ΚΛΑΔΟΣ_ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ` = s.`ΚΛΑΔΟΣ ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ`
                                           ,t.`ΝΟΜΙΚΗ_ΜΟΡΦΗ_ΕΤΑΙΡΕΙΑΣ` = s.`ΝΟΜΙΚΗ ΜΟΡΦΗ ΕΤΑΙΡΕΙΑΣ`
""")
#display(merge_tiresias_customer_1)  

In [0]:
merge_tiresias_customer_2=spark.sql(f"""MERGE INTO `{env2}-cdh-delivery`.tiresias.customer AS t USING (
										WITH address_tokens AS (
											SELECT t.cdi
												,t.portfolioname
												,t.address_name
												,CASE 
													WHEN t.address_name NOT RLIKE '[0-9]' THEN ''
													WHEN LTRIM(RTRIM(RIGHT(t.address_name, INSTR(REVERSE(t.address_name), ' ') - 1))) NOT RLIKE '[0-9]' THEN ''
													ELSE LTRIM(RTRIM(RIGHT(t.address_name, INSTR(REVERSE(t.address_name), ' ') - 1)))
												END AS last_token
											FROM `{env2}-cdh-delivery`.tiresias.customer t
											JOIN customerdata cc
												ON cc.cdi = t.cdi 
												AND cc.portfolioname = t.portfolioname
										)
										SELECT cdi
											,portfolioname
											,address_name
											,REPLACE(address_name, last_token, '') AS newOdos
											,last_token AS `ΑΡΙΘΜΟΣ`
										FROM address_tokens
										WHERE last_token <> ''
									) s 
           								ON (s.cdi=t.cdi AND t.portfolioname=s.portfolioname)
										WHEN MATCHED THEN UPDATE SET 
										 t.address_name =  s.newOdos 
										,t.address_number =  s.`ΑΡΙΘΜΟΣ`
""")
#display(merge_tiresias_customer_2)  


In [0]:
#insert into #tiresias_loan
insert_into_tiresias_loan=spark.sql(f"""
                                    	INSERT INTO `{env2}-cdh-delivery`.tiresias.loan (
													 portfolioName
													, account_number
													,`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
													, product_key
													, loan_status
													, Tiresias_File 
													, migration_date
													, tiresias_code_init 
													, approval_date
													, approval_amn
													, origination_amn
													, Last_disbursed_date
													, AGREEMENT_NUMBER
													, maturity_date
													, installments
													, install_frq
													, is_open_loan 
													,`ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`
													,`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠ_ΔΑΝΕΙΟΥ`
													,`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠ_ΔΑΝΕΙΟΥ`
													,`ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠ_ΔΑΝΕΙΟΥ`
													,`RRE_ΓΙΑ_ΑΠΦ`

												)

											SELECT    a.portfolioName
													, a.account_number
													, a.tiresias_account_number
													, a.product_key as `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
													, a.exposurestatus
													, a.tiresiasfile 
													, a.migrationdate
													, a.tiresias_code_init   as tiresias_code_init
													, a.origination_date as  approval_date
													, a.origination_amount   as approval_amn
													, a.origination_amount
													, a.origination_date
												    , a.agreement_number
													, a.maturitydate
													, a.NumberOfInstallments
													, a.InstalmentFrequency
													, CASE WHEN a.tiresias_code_init  IN ('5', '3', 'K', 'R', 'V') THEN 1 ELSE 0 END is_open_loan
													,`ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`              -- New Addition ASK-2976 ΠΕΕ EG 20250218
													,`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`-- New Addition ASK-2976 ΠΕΕ EG 20250218
													,`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`-- New Addition ASK-2976 ΠΕΕ EG 20250218
													,`ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`    -- New Addition ASK-2976 ΠΕΕ EG 20250218
													,`RRE_ΓΙΑ_ΑΠΦ`  -- New Addition ASK-2976 ΠΕΕ EG 20250218 
											FROM `{env2}-cdh-delivery`.tiresias.loan_data a  
											WHERE NOT EXISTS (
													SELECT 1
													FROM  `{env2}-cdh-delivery`.tiresias.loan s
													WHERE s.portfolioName = a.portfolioName AND s.account_number = a.account_number 
													)
    							""")
#display(insert_into_tiresias_loan) 

In [0]:

update_filetype_code=spark.sql(f""" MERGE INTO `{env2}-cdh-delivery`.tiresias.loan AS t USING (
										SELECT  tiresias_code_init
          										,tiresiasfile
                    							,account_number
                           						,ll.PortfolioName
										FROM `{env2}-cdh-delivery`.tiresias.loan_data ll
										) s
										ON (s.account_number = t.account_number AND t.PORTFOLIONAME  = s.PORTFOLIONAME)
									WHEN MATCHED THEN
											UPDATE	SET t.tiresias_code_init = s.tiresias_code_init
											           ,t.tiresias_file= s.tiresiasfile
    """)
    
#display(update_filetype_code)  

In [0]:

merge_tiresias_loan=spark.sql(f"""MERGE INTO `{env2}-cdh-delivery`.tiresias.loan AS t USING (
											SELECT   ll.ACCOUNT_NUMBER
													,ll.`ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`
													,ll.portfolioid
													,ll.PORTFOLIONAME
													,ll.`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
													,ll.`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
													,ll.`ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
													,`RRE_ΓΙΑ_ΑΠΦ`
											FROM `{env2}-cdh-delivery`.tiresias.loan_data ll
											) s
											ON (s.account_number = t.account_number AND t.PORTFOLIONAME  = s.PORTFOLIONAME)
										WHEN MATCHED THEN UPDATE
												SET  t.`ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ` = s.`ΔΕΙΚΤΗΣ_ΚΑΛΥΨΗΣ_ΧΟΡΗΓΗΣΗΣ`
													,t.`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠ_ΔΑΝΕΙΟΥ`= s.`ΑΡΙΘΜΟΣ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
													,t.`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠ_ΔΑΝΕΙΟΥ` =s.`ΣΥΝΟΛΙΚΟ_ΕΓΚΡΙΘΕΝ_ΠΟΣΟ_ΣΥΜΒΑΣΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
													,t.`ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠ_ΔΑΝΕΙΟΥ`=s.`ΔΕΙΚΤΗΣ_ΠΟΣΟΣΤΟΥ_ΣΥΜΜΕΤΟΧΗΣ_ΚΟΙΝΟΠΡΑΚΤΙΚΟΥ_ΔΑΝΕΙΟΥ`
												    ,t.`RRE_ΓΙΑ_ΑΠΦ`=s.`RRE_ΓΙΑ_ΑΠΦ`
    """)
    
#display(merge_tiresias_loan)  

In [0]:
correct_rel_accounts=spark.sql(f"""
                                    MERGE INTO `{env2}-cdh-delivery`.tiresias.loan AS target
                                    USING (
                                        SELECT loan.account_number ,loan.portfolioName
                                        FROM  `{env2}-cdh-delivery`.tiresias.loan loan
                                        JOIN  `{env2}-cdh-delivery`.tiresias.loan_data x 
                                        ON x.account_number = loan.account_number
                                        and loan.portfolioname=x.portfolioName
                                        WHERE  x.is_slave = FALSE
                                    ) AS source
                                    ON  target.account_number = source.account_number
                                    AND target.portfolioName = source.PortfolioName
                                    AND target.isSlave = TRUE
                                    AND target.isActive = true
                                    WHEN MATCHED THEN
                                    UPDATE SET    target.isSlave = false
                                                , target.MasterAccount = null
""")

#display(correct_rel_accounts) 



In [0]:
 
merge_tiresias_loan_2=spark.sql(f"""MERGE INTO  `{env2}-cdh-delivery`.tiresias.loan AS t USING (
                                          SELECT l.portfolioName
                                                ,l.account_number
                                                ,TRUE slave
                                                ,m.account_number MasterAccount
                                      FROM `{env2}-cdh-delivery`.tiresias.loan_data s
                                      JOIN `{env2}-cdh-delivery`.tiresias.loan l
                                      ON l.account_number=s.account_number
                                      AND l.portfolioname=s.portfolioname
                                      JOIN `{env2}-cdh-delivery`.tiresias.loan_data m
                                      ON   m.profits_Account_Number=s.masterAccount  
                                      WHERE s.is_slave = TRUE 
                                      AND COALESCE(l.isSlave,FALSE) <> TRUE
                                    ) s 
                                    ON s.portfolioName=t.portfolioName AND s.account_number=t.account_number 
                                    WHEN MATCHED THEN
                                    UPDATE SET
                                     t.isSlave=s.slave
                                    ,t. masterAccount=s.MasterAccount;
  """)


#display(merge_tiresias_loan_2)  


In [0]:
merge_tiresias_loan_3=spark.sql(f"""MERGE INTO`{env2}-cdh-delivery`.tiresias.loan AS t USING (
											SELECT  l.account_number
           										    ,l.portfolioname
                       								, m.`approval_date`
                               						, 0.01 `approval_amn`
                                     				, m.`Last_disbursed_date`
                                         			, m.`denounce_date`
                                            		, m.`maturity_date`
                                              		, 0.01 `origination_amn`
                                                	, m.tiresias_code_init
                                                 	, m.is_open_loan 
											FROM `{env2}-cdh-delivery`.tiresias.loan m
											JOIN `{env2}-cdh-delivery`.tiresias.loan l 
           									ON m.account_number=l.MasterAccount 
                    						AND m.portfolioname=l.portfolioname
											WHERE  l.isslave = TRUE 
											EXCEPT 
											SELECT  l.account_number
           											,l.portfolioname
                      								, l.approval_date
                              						, l.approval_amn
                                    				, l.last_disbursed_date
                                        			, l.denounce_date
                                           			, l.maturity_date
                                              		, origination_amn
                                                	, l.tiresias_code_init
                                                 	, l.is_open_loan
											from `{env2}-cdh-delivery`.tiresias.loan l
											where l.isslave = TRUE

										) s 
          								ON s.account_number=t.account_number AND s.portfolioname=t.portfolioname
										WHEN MATCHED THEN 
          								UPDATE SET    t.`approval_date`= s.`approval_date`
                  									,t.`approval_amn`  = s.`approval_amn`
                           							,t.`Last_disbursed_date`= s.`Last_disbursed_date`
												    ,t.`denounce_date` = s.`denounce_date`
                									,t.`maturity_date` = s.`maturity_date` 
                         							,t.`origination_amn`=s.`origination_amn`
                                					,t.tiresias_code_init = s.tiresias_code_init
                                     				,t.is_open_loan  = s.is_open_loan ;
                  """)

#display(merge_tiresias_loan_3)   



In [0]:

merge_tiresias_loan_customer=spark.sql(f"""MERGE INTO  `{env2}-cdh-delivery`.tiresias.loan_customer AS t USING (
													SELECT  cl.portfolioname
             												,cl.account_number
															,cl.cdi
               												,cl.partyrelationshiptype AS relation
															, CASE WHEN cl.partyrelationshiptype = 'Primary Owner' THEN 1 
               														WHEN cl.partyrelationshiptype = 'Co-Owner' THEN 3 
                             										WHEN cl.partyrelationshiptype = 'Guarantor' THEN 2
                                       						 END `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`
                                              				,TiresiasFile AS Tiresias_file 
															,cl.NationalNumber  AS id_doc
															,cl.Partytaxregistrationnumber AS taxnumber
															,l.tiresias_account_number tiresias_account_code
													FROM loancustomerdata cl
													INNER JOIN `{env2}-cdh-delivery`.tiresias.loan_data l
             										ON l.account_number = cl.account_number 
                       								AND cl.portfolioname = l.portfolioname
													
													EXCEPT
													
													SELECT   cl.portfolioname
             												,cl.account_number
                         									,cl.cdi
                                  							,cl.relation
                                         					,cl.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`
                                              				,cl.Tiresias_file
                                                  			,cl.id_doc
                                                     		,cl.tax_number
                                                            ,cl.tiresias_account_code
													FROM `{env2}-cdh-delivery`.tiresias.loan_customer cl
													) s
													ON (s.portfolioname = t.portfolioname AND s.account_number = t.account_number AND s.cdi = t.cdi)
												WHEN NOT MATCHED THEN	
														INSERT (portfolioname, account_number, cdi, relation, `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,Tiresias_file,
																id_doc, tax_number, tiresias_account_code, tiresias_export_flg, send_as_closed_relation
                												)
														VALUES (s.portfolioname, s.account_number,s.cdi,s.relation,s.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,s.Tiresias_file,
																s.id_doc, s.taxnumber, s.tiresias_account_code, 1, FALSE
                												) 
    
    """)

#display(merge_tiresias_loan_customer)  

In [0]:
merge_tiresias_loan_customer_part_b=spark.sql(f"""MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_customer AS t USING (
															SELECT    cl.portfolioname
               														, cl.account_number
																	, cl.cdi
                 													, cl.partyrelationshiptype AS relation
																	, CASE  WHEN cl.partyrelationshiptype = 'Primary Owner' THEN 1 
                 															WHEN cl.partyrelationshiptype = 'Co-Owner' THEN 3 
                                											WHEN cl.partyrelationshiptype = 'Guarantor' THEN 2 
                                           								END AS `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`
                                                   					, TiresiasFile AS Tiresias_file
																	, cl.NationalNumber  AS id_doc
																	, cl.Partytaxregistrationnumber AS taxnumber
																	, l.tiresias_account_number AS tiresias_account_code
															FROM loancustomerdata cl
															INNER JOIN `{env2}-cdh-delivery`.tiresias.loan_data l
               												 ON l.account_number = cl.account_number
                            								 AND cl.portfolioname = l.portfolioname 
                                     
															EXCEPT
															
															SELECT cl.portfolioname
               													 , cl.account_number
                             									, cl.cdi
                                      							, cl.relation
                                             					, cl.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`
                                                  				,cl.Tiresias_file
                                                      			,cl.id_doc
                                                         		,cl.tax_number
                                                           		, cl.tiresias_account_code
															FROM `{env2}-cdh-delivery`.tiresias.loan_customer cl
															) s
															ON (s.portfolioname = t.portfolioname AND s.account_number = t.account_number AND s.cdi = t.cdi)
												
												WHEN MATCHED and t.tiresias_export_flg <> 0  THEN
													UPDATE SET  t.relation = s.relation,
																t.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ` = s.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`,
																t.tax_number = COALESCE(s.taxnumber, t.tax_number),
																t.id_doc = COALESCE(s.id_doc, t.id_doc),
																t.tiresias_account_code = s.tiresias_account_code,
																t.Tiresias_file = s.Tiresias_file, 
																t.updated = CURRENT_TIMESTAMP
               """)

#display(merge_tiresias_loan_customer_part_b)  
   

In [0]:
#CLOSEDrelation
closed_relation=spark.sql(f""" 
                            SELECT relation, CDI, account_number, c.portfolioName
                            FROM  `{env2}-cdh-delivery`.tiresias.loan_customer c
                            WHERE Tiresias_export_flg= 1
                            AND EXISTS (SELECT 1
                                        FROM `{env2}-cdh-delivery`.tiresias.loan_data x
                                        WHERE x.account_number=c.account_number 
                                        AND x.portfolioname=c.portfolioName
                                        )
                            AND remove_relation_dt IS NULL
                            EXCEPT 
                            SELECT partyrelationshiptype, CDI, account_number, portfolioName 
                            FROM   LoanCustomerData
""")
closed_relation.createOrReplaceTempView("closed_relation")
 

In [0]:
remove_relation_upd=spark.sql(f"""
                                MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_customer AS lc 
                                USING closed_relation  AS c
                                ON lc.account_number = c.account_number 
                                AND lc.cdi = c.cdi
                                AND lc.portfolioName = c.portfolioName
                                AND lc.relation = c.relation
                                WHEN MATCHED AND lc.remove_relation_dt IS NULL THEN
                                    UPDATE SET
                                        lc.remove_relation_dt = CURRENT_TIMESTAMP,
                                        lc.remove_relation_reason = 'relation removed',
                                        lc.updated = CURRENT_TIMESTAMP
                                 """)
#display(remove_relation_upd)  

In [0]:
merge_tiresias_customer_3=spark.sql(f"""
                                    	MERGE INTO  `{env2}-cdh-delivery`.tiresias.customer t USING (
											SELECT DISTINCT cdi, id_doc, tax_number
											FROM (
													SELECT cl.cdi, cl.id_doc, cl.tax_number
             												,RANK() OVER(PARTITION BY cl.cdi ORDER BY  CASE WHEN tax_Number IS NOT NULL 
             																										AND id_doc IS NOT NULL THEN 0 
																											WHEN tax_Number IS NOT NULL
                                   																					AND id_doc IS NULL THEN 1 
                                                                											ELSE 2 END 
                                                                        ) rnk
													FROM `{env2}-cdh-delivery`.tiresias.loan_customer cl
													WHERE tiresias_export_flg <> 0 --or tiresias_export_flg is null
												) AS d WHERE d.rnk=1
			
											EXCEPT
			
											SELECT  c.cdi, c.`ADT`, c.`AFM`
											FROM `{env2}-cdh-delivery`.tiresias.customer c
											) s
										ON ( s.cdi = t.cdi)
										WHEN MATCHED THEN
          							    UPDATE SET    t.`AFM` = COALESCE(s.tax_number, t.`AFM`)
                     								, t.`ADT` = COALESCE(s.id_doc,t.`ADT`)
                             						, t.updated = current_date
  
  """)
#display(merge_tiresias_customer_3)   

In [0]:
#VALIDATION for AFM number
from pyspark.sql.types import BooleanType


def is_valid_tax_number(Input : str):
    if Input is None:
        return False
    elif not Input.isdigit():
        return False
    elif Input == '999999993':
        return False
    elif any(c in Input for c in ['+', '-', '.', ',', 'E', 'e']):
        return False 
    elif len(Input) == 8:
        Input = '0' + Input
    if len(Input) != 9:
        return False
    factor = 256
    check_digit = int(Input[8])
    value =0
    for i in range(0,8):
        digit = int(Input[i])
        value = value + digit * factor
        factor = factor // 2
    if value == 0:
        return False
    checksum = value % 11
    if checksum == 10:
        checksum=0
    if checksum == check_digit:
        return True
    else:
        return False

# Convert your Python function into a UDF
is_valid_tax_number_udf = udf(is_valid_tax_number, BooleanType())
spark.udf.register("is_valid_tax_number_udf", is_valid_tax_number_udf)

In [0]:
validate_afm_adt=spark.sql(f"""
                            UPDATE `{env2}-cdh-delivery`.tiresias.customer 
                            SET     hasValidTaxNumber= is_valid_tax_number_udf(AFM) 
                                    ,hasValidDocID= CASE
                                        WHEN ADT IS NULL OR ADT = '' OR LENGTH(ADT) < 7 OR LENGTH(ADT) > 8 THEN FALSE
                                        WHEN LENGTH(ADT) = 7 AND ADT RLIKE '^[A-Za-zΑ-Ωα-ω][0-9]{{6}}$' THEN TRUE
                                        WHEN LENGTH(ADT) = 8 AND ADT RLIKE '^[A-Za-zΑ-Ωα-ω]{{2}}[0-9]{{6}}$' THEN TRUE
                                        ELSE FALSE
                                    END
""")
#display(validate_afm_adt) 

In [0]:
update_valid_Afm_Adt_reason=spark.sql(f"""
                                      UPDATE `{env2}-cdh-delivery`.tiresias.customer
                                      SET  isValid = FALSE
                                         , Invalid_reason = 'no afm and adt'
                                      WHERE hasValidTaxNumber = FALSE
                                      AND hasValidDocID = FALSE 
                                      AND isValid != FALSE 
                                      AND Invalid_reason IS NULL
                                    """)

#display(update_valid_Afm_Adt_reason)  

In [0]:
update_valid_Afm_Adt_reason_2=spark.sql(f"""
                                      UPDATE `{env2}-cdh-delivery`.tiresias.customer 
                                      SET   isValid = TRUE
                                          , Invalid_reason = NULL
                                      WHERE (hasValidTaxNumber = TRUE OR hasValidDocID = TRUE) 
                                      AND isValid = FALSE
                                      AND Invalid_reason = 'no afm and adt'
  """)

#display(update_valid_Afm_Adt_reason_2)   

In [0]:
update_valid_Afm_Adt_reason_3=spark.sql(f"""
										UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer 
										SET   Tiresias_export_flg = 1
          									, Tiresias_not_sent_reason = NULL
                   							, updated = current_timestamp
										WHERE Tiresias_export_flg NOT IN (0, 1)
          								AND Tiresias_not_sent_reason = 'no afm and adt' 
                   						AND NOT EXISTS (
												SELECT 1
												FROM `{env2}-cdh-delivery`.tiresias.customer c
												WHERE c.cdi = loan_customer.cdi AND c.Invalid_reason = 'no afm and adt' 
            									AND isValid = FALSE
												)
    """)
#display(update_valid_Afm_Adt_reason_3) 

In [0]:
update_valid_Afm_Adt_reason_4=spark.sql(f"""
                                    UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer
									SET   Tiresias_export_flg = 1
         								, Tiresias_not_sent_reason = NULL
                 						, updated = CURRENT_TIMESTAMP
									WHERE Tiresias_export_flg NOT IN (0, 1)
         							AND Tiresias_not_sent_reason = 'no afm and adt' 
                					AND NOT EXISTS (
											SELECT 1
											FROM `{env2}-cdh-delivery`.tiresias.customer c
											WHERE c.cdi = loan_customer.cdi
           									AND c.Invalid_reason = 'no afm and adt' 
                     						AND isValid = FALSE
           								)
    """)


#display(update_valid_Afm_Adt_reason_4) 

In [0]:
update_valid_Afm_Adt_reason_5 =spark.sql(f"""
                                    UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer
									SET   Tiresias_export_flg = 2
         								, Tiresias_not_sent_reason = 'no afm and adt'
                 						, updated = CURRENT_TIMESTAMP
									WHERE Tiresias_export_flg = 1 
         							AND EXISTS (
											SELECT 1
											FROM `{env2}-cdh-delivery`.tiresias.customer c
											WHERE c.cdi = loan_customer.cdi 
           									AND c.Invalid_reason = 'no afm and adt'
                    					    AND isValid = FALSE
           							)
    """)

#display(update_valid_Afm_Adt_reason_5)


In [0]:
duplicate_relations_update=spark.sql(f"""
                                MERGE INTO `{env2}-cdh-delivery`.tiresias.loan_customer cust  USING (
										SELECT b
										FROM (
												SELECT    concat(a.account_number,a.cdi) as b
            											, rank() OVER (	PARTITION BY a.tiresias_account_code, COALESCE(a.tax_number, ''), COALESCE(a.id_doc, '') 
                       													ORDER BY `ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ` DESC, concat(a.account_number,a.cdi)
								 						) rnk
												FROM  `{env2}-cdh-delivery`.tiresias.loan_customer  a
												INNER JOIN (
													SELECT portfolioname, tiresias_account_code, COALESCE(tax_number, '') tax_number, COALESCE(id_doc, '') id_doc
													FROM  `{env2}-cdh-delivery`.tiresias.loan_customer 
													WHERE Tiresias_export_flg = 1
													GROUP BY portfolioname, tiresias_account_code, tax_number, id_doc
													HAVING count(1) > 1
													) AS d 
             									ON  d.portfolioname = a.portfolioname
                      							AND d.tiresias_account_code = a.tiresias_account_code 
                             					AND d.tax_number = COALESCE(a.tax_number, '') 
                                  				AND d.id_doc = COALESCE(a.id_doc, '')
												WHERE a.Tiresias_export_flg = 1
										) s
										WHERE rnk = 1  
          						) k  ON (k.b=concat(cust.account_number, cust.cdi))
								WHEN MATCHED THEN
								UPDATE SET    Tiresias_export_flg = 2
        									, Tiresias_not_sent_reason = 'duplicated relation'
                 							, updated = CURRENT_TIMESTAMP
		""")

#display(duplicate_relations_update)

In [0]:
no_loan_status_update=spark.sql(f"""
						UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer 
						SET   Tiresias_export_flg = 2
      						, Tiresias_not_sent_reason = 'no loan_status'
           					, updated = CURRENT_TIMESTAMP
						WHERE Tiresias_export_flg = 1
      					AND account_number IN (
										SELECT account_number
										FROM  `{env2}-cdh-delivery`.tiresias.loan
										WHERE Tiresias_export_flg = 1 
          								AND   loan_status IS NULL 
                  						AND   isActive = TRUE
                        			)
    """)

#display(no_loan_status_update)  

In [0]:
no_product_update=spark.sql(f"""
						UPDATE `{env2}-cdh-delivery`.tiresias.loan_customer 
						SET   Tiresias_export_flg = 2
      						, Tiresias_not_sent_reason = 'no product found in loan'
            				, updated = CURRENT_TIMESTAMP
						WHERE Tiresias_export_flg = 1 
      					AND account_number IN (
										SELECT account_number
										FROM `{env2}-cdh-delivery`.tiresias.loan
										WHERE  Tiresias_export_flg = 1 
          								AND product_key IS NULL
										)
    """)

#display(no_product_update)  


In [0]:
no_loan_status_update_2=spark.sql(f"""
							UPDATE `{env2}-cdh-delivery`.tiresias.loan 
							SET   Tiresias_export_flg = 2
       							, Tiresias_not_sent_reason = 'no loan_status'
              					, updated = CURRENT_TIMESTAMP
							WHERE Tiresias_export_flg = 1 
							AND loan_status IS NULL 
							AND isActive = TRUE
  """)

#display(no_loan_status_update_2) 

In [0]:
no_product_update_2=spark.sql(f"""
						UPDATE `{env2}-cdh-delivery`.tiresias.loan 
						SET   Tiresias_export_flg = 2
      						, Tiresias_not_sent_reason = 'no product found in loan'
            				, updated = CURRENT_TIMESTAMP
						WHERE Tiresias_export_flg = 1 
						AND   product_key IS NULL
  """)

#display(no_product_update_2)  

In [0]:
merge_loan_1=spark.sql(f"""
					MERGE INTO `{env2}-cdh-delivery`.tiresias.loan t USING (
         					SELECT p.account_number
								, p.portfolioname 
								, l.eomdate  EOM_DATE
								, COALESCE(l.`remaining_installments`, 0) remaining_installments
								, l.`dpd` dpd
								, COALESCE(l.`InstallmentAmount`, 0) install_amn
        						, l.NumberOfInstallments  
								, CASE  WHEN l.isactiveflag = False THEN 0
        								WHEN l.NORMAL_BALANCE < 0 THEN 0 
                						ELSE l.NORMAL_BALANCE 
                      				END NRM_BAL
								, CASE 	WHEN l.isactiveflag = FALSE THEN 0 
        							  	WHEN l.OVERDUE_BALANCE < 0 THEN 0 
                  					    ELSE l.OVERDUE_BALANCE 
                           			END OV_BAL
								, l.exposurestatus loan_status
								, l.isactiveflag isactive
								, l.closeddate closed_date 
								, l.impl_type
								, CASE  WHEN l.impl_type IS NOT NULL THEN 1
        								ELSE 0 
                					END is_modified
								, l.last_payment_date
								, l.total_payment_amount total_payment_amn
								, l.prv_month_payment_amount prv_month_payment_amn
							FROM  `{env2}-cdh-delivery`.tiresias.loan p
							CROSS JOIN param_dates dt
							INNER JOIN `{env2}-cdh-delivery`.tiresias.loan_data l 
       						ON l.account_number = p.account_number 
             				AND l.portfolioname = p.portfolioname
					) s ON (s.account_number = t.account_number AND t.portfolioname = s.portfolioname AND t.Tiresias_export_flg = 1)
					WHEN MATCHED THEN
					UPDATE  SET   t.install_amn = s.install_amn
     							, t.installments=s.NumberOfInstallments
								, t.dpd = s.dpd
								, t.NRM_BAL = s.NRM_BAL
								, t.OV_BAL = s.OV_BAL
								, t.LOAN_STATUS = s.LOAN_STATUS
								, t.remaining_install = CASE WHEN s.remaining_installments > t.installments THEN t.installments ELSE s.remaining_installments END
								, t.closed_date = CASE WHEN s.isactive=FALSE then COALESCE(s.closed_date, s.EOM_DATE) END    
								, t.updated = CURRENT_TIMESTAMP
								, t.tiresias_last_export_dt = s.eom_date
								, t.tot_bal = s.NRM_BAL + s.OV_BAL
								, t.impl_type = s.impl_type
								, t.is_modified = s.is_modified
								, t.last_payment_date = s.last_payment_date
								, t.total_payment_amn = s.total_payment_amn
								, t.prv_month_payment_amn = s.prv_month_payment_amn
								, t.isactive = s.isactive 
        """)

#display(merge_loan_1) 

In [0]:
loan_fixes_upd=spark.sql(f"""
                UPDATE  `{env2}-cdh-delivery`.tiresias.Loan 
                SET   installments=0 
                    , remaining_install=0 
                    , install_frq=0
                    , install_amn=0
                WHERE is_open_loan = TRUE
                AND isActive = TRUE
                AND Tiresias_File='B'
                AND tiresias_code_mod='K'
                AND (installments<>0 or remaining_install<>0 or install_frq<>0 or install_amn<>0)
""")

#display(loan_fixes_upd)

In [0]:
#modifications cl%
merge_loan_2=spark.sql(f"""
				MERGE INTO `{env2}-cdh-delivery`.tiresias.Loan  t  USING  (
							SELECT l.ACCOUNT_NUMBER
								, l.portfolioname
								, right(concat('00',ll.impl_pee),2) `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ` 
								,'Μ' `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ` 
								, CASE  WHEN ll.UnderLawType IS NOT NULL THEN '1'
        								ELSE '0' 
                				END `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`
								, l.approval_date `ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ`
								, CASE  WHEN l.impl_type = 'CL-mod' THEN 0 
        								WHEN l.impl_type in ('CL-woff', 'CL-slv') AND l.maturity_date > dt.eom_Date THEN '1'
                						ELSE '2' 
                      			END `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` 
								--check transactions  look document
								, CASE  WHEN l.impl_type = 'CL-mod' THEN NULL 
        								WHEN l.impl_type in ('CL-woff', 'CL-slv')  THEN '2' 
                						ELSE NULL
                      			END `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
								, CASE  WHEN l.impl_type in ('CL-woff', 'CL-slv') AND l.maturity_date < dt.eom_Date THEN l.maturity_date 
        								ELSE NULL 
                				END `ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
								, 1 `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`
								, substring(l.agreement_number, 1, 15) `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`
								, '1'  `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`
								, `{env2}-cdh-delivery`.tiresias.GetBucketMod(l.dpd, l.loan_status) modification_bucket_flag  
								, CASE  WHEN l.tiresias_file = 'B' THEN 'O' 
        								WHEN l.tiresias_file = 'C' AND ll.is_housing = TRUE THEN 'F' 
                						ELSE 'B' 
                      			END tiresias_code_mod
								, FALSE is_open_loan --2022-05-21
							FROM  `{env2}-cdh-delivery`.tiresias.Loan l
							INNER JOIN  `{env2}-cdh-delivery`.tiresias.loan_data ll
       						ON ll.account_number = l.account_number
             				AND ll.portfolioname = l.portfolioname
							CROSS JOIN param_dates dt
							WHERE l.is_modified = TRUE 
       						AND l.Tiresias_export_flg = 1 
             				AND l.impl_type IN ('CL-mod', 'CL-woff', 'CL-slv') 

							UNION --CL_CLOSED
							
							SELECT    l.ACCOUNT_NUMBER
									, l.portfolioname
									, right(concat('00',ll.impl_pee),2)  `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`
									,'Μ' `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ` 
									, CASE WHEN ll.UnderLawType IS NOT NULL THEN '1'
         									ELSE '0' 
                  					END `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ` 
									, l.closed_date `ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ`
									, '1' `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` --check transactions  look document	 --because of the exception  
									, '2' `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` --because of the exception
									, NULL `ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
									, 1 `ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`
									, substring(l.agreement_number, 1, 15) `ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`
									,'0'  `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`
									,try_cast(' ' AS BIGINT) modification_bucket_flag 
									, NULL tiresias_code_mod
									,l.is_open_loan  --2022-05-21
							FROM `{env2}-cdh-delivery`.tiresias.Loan l
							INNER JOIN `{env2}-cdh-delivery`.tiresias.loan_data ll 
       						ON ll.account_number = l.account_number 
             				AND ll.portfolioname = l.portfolioname
							CROSS JOIN param_dates dt
							WHERE   l.is_modified = TRUE
       						AND l.Tiresias_export_flg = 1
              				AND l.impl_type IN ('CL-CLOSED')
				) s ON (s.account_number = t.account_number AND t.portfolioname = s.portfolioname)
				WHEN MATCHED THEN
				UPDATE	SET 
								t.`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ` = s.`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`
								,t.`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`  = s.`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ` 
								, t.`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ` = s.`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`
								, t.`ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ` = s.`ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ`
								, t.`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = s.`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
								, t.`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = s.`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
								, t.`ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = s.`ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
								, t.`ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ` = s.`ΔΕΙΚΤΗΣ_ΥΠΑΡΞΗΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`
								, t.`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ` = s.`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ`
								, t.`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ` = s.`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`
								, T.modification_bucket_flag = S.modification_bucket_flag
								, t.tiresias_code_mod = s.tiresias_code_mod
								,t.is_open_loan=s.is_open_loan 
            """)

#display(merge_loan_2) 

In [0]:
merge_loan_3=spark.sql(f""" 
				MERGE INTO  `{env2}-cdh-delivery`.tiresias.Loan t USING (
					SELECT    a.ACCOUNT_NUMBER
							, a.portfolioname
							, ll.settlementStatus
							, ll.settlementExpiryDate
							, '17'`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ` --17  is settlement exists διακανονισμός Απαιτήσεων ισ σεττλεμεντ
							, 'Ο' `ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ` -- 'O'  -- greek O when settlement exists
							, CASE WHEN ll.UnderLawType IS NOT NULL THEN '1'
       								ELSE '0'
               				END `ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ` ---look document  3869 1 else 0    where peee null end
							, ll.settlementImplementationDate `ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ`
							, CASE WHEN ll.settlementStatus IN ('Fulfilled', 'Out of Collection') AND a.isactive = FALSE AND a.tot_bal = 0 THEN 2
       								ELSE 1 
               				END `ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
							, '2' `ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
							, CASE WHEN ll.settlementStatus IN ('Fulfilled', 'Out of Collection') AND a.isactive = FALSE AND a.tot_bal = 0 THEN dt.eom_lst_wrk_date
       								ELSE NULL
               				 END `ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
							, '0' `ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`
							, CASE WHEN ll.settlementoverduedate IS NOT NULL 
       									AND ll.settlementoverduedate <= dt.eom_lst_wrk_date 
                						THEN  `{env2}-cdh-delivery`.tiresias.GetBucketMod(datediff(ll.settlementoverduedate, dt.eom_lst_wrk_date), '1') 
                					ELSE '0'
                     		END as modification_bucket_flag
					FROM `{env2}-cdh-delivery`.tiresias.Loan a
					INNER JOIN `{env2}-cdh-delivery`.tiresias.Loan_data ll 
     				ON ll.account_number = a.account_number 
         			AND ll.portfolioname = a.portfolioname
					CROSS JOIN  param_dates dt
					WHERE a.Tiresias_export_flg = 1 
     				AND a.impl_type = 'settlement' 
         			AND a.is_modified = TRUE 
			) s ON (s.account_number = t.account_number AND t.portfolioname = s.portfolioname)
				WHEN MATCHED	THEN
				UPDATE SET t.`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ` = s.`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ`
						, t.`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ` = s.`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ`
						, t.`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ` = s.`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ`
						, t.`ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ` = s.`ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ`
						, t.`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = s.`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
						, t.`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = s.`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
						, t.`ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = s.`ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`
						, t.`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`= s.`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ`
						, t.modification_bucket_flag = s.modification_bucket_flag
						,t.tiresias_code_mod=t.tiresias_code_init 
      """)

#display(merge_loan_3)  

In [0]:
final_loan_merge=spark.sql(f"""	
                MERGE INTO `{env2}-cdh-delivery`.tiresias.Loan t	USING (
						SELECT a.ACCOUNT_NUMBER, a.portfolioname
						FROM `{env2}-cdh-delivery`.tiresias.Loan a
						WHERE `ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ` IS NOT NULL
      					AND a.Tiresias_export_flg = 1 
            			AND a.is_modified = FALSE
						and a.isactive=true
				) s ON (s.account_number = t.account_number AND t.portfolioname = s.portfolioname)
				WHEN MATCHED THEN
				UPDATE	SET  t.`ΚΑΤΑΤΑΞΗ_ΠΕΕ_ΒΑΣΕΙ_ΠΕΕ` = NULL
							, t.`ΕΙΔΟΣ_ΡΥΘΜΙΣΗΣ` = NULL
							, t.`ΘΕΣΜΙΚΟ_ΠΛΑΙΣΙΟ` = NULL
							, t.`ΗΜ_ΝΙΑ_ΡΥΘΜΙΣΗΣ` = NULL
							, t.`ΕΝΔΕΙΞΗ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ`= NULL
							, t.`ΕΙΔΟΣ_ΠΡΟΓΡΑΜΜΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = NULL
							, t.`ΗΜ_ΝΙΑ_ΑΦΕΣΗΣ_ΧΡΕΟΥΣ` = NULL
							, t.`ΚΑΤΑΣΤΑΣΗ_ΡΥΘΜΙΣΗΣ` = NULL
							, t.modification_bucket_flag = NULL
							, t.tiresias_code_mod = NULL
    """)
#display(final_loan_merge)

